# DME Express — Operations Summary Dashboard

**Version v1.0.0 — 2026-08-14** | companion to `tech_workload` v1.35.0

Produces, for the **company**, each **VP**, each **metro**, and each **warehouse**:
technician productivity over time (weekly, rolling 4-week average), lost equipment
over time (monthly), redeliveries over time (monthly, with per-100-ticket rates),
stock outs over time (**pending source table — config-gated**), and disjoint
top/bottom technician lists with statistics.

**Outputs:** `OpsDashboard_Pages_{date}.pdf` (one page per entity, board reading
order: company → VPs → metros → warehouses, ~85 pages), per-VP/company PNGs, and
`OpsDashboard_Report_{date}.xlsx` (17+ sheets).

**Architecture (decided 2026-08-14):** self-contained — re-queries Azure SQL via the
Key Vault login and duplicates the v1.35.0 extraction/matching machinery. Duplicated
cells are marked LIFTED VERBATIM; when the main notebook's matching changes, re-sync
them or the two reports will diverge. Payroll (PLC) and APC census are deliberately
NOT pulled: no metric here needs hours, so the dashboard runs faster and carries no
PTO-contamination caveat.

## Cell 1 — INSTALL DEPENDENCIES

In [1]:
%pip install rapidfuzz matplotlib python-dotenv pypyodbc openpyxl python-dateutil azure-identity azure-keyvault-secrets

Note: you may need to restart the kernel to use updated packages.


## Cell 2 — IMPORTS

In [2]:
# Standard library + analysis stack. azure.identity/azure.keyvault power the Key
# Vault login in Cell 4. rapidfuzz and statsmodels are optional: flags _RAPIDFUZZ_OK
# and _SM_OK let later cells degrade gracefully instead of crashing on import.
import os, re, time, warnings, calendar as _cal
from datetime import datetime, date, timedelta
from collections import defaultdict
from functools import reduce
from dateutil.relativedelta import relativedelta
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
from dotenv import load_dotenv
import pypyodbc as odbc
import scipy.stats as stats
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient

try:
    from rapidfuzz.distance import DamerauLevenshtein as _DL
    _RAPIDFUZZ_OK = True
except ImportError:
    _RAPIDFUZZ_OK = False
    print('rapidfuzz not installed — falling back to pure-Python Levenshtein.')
try:
    import statsmodels.api as sm; _SM_OK = True
except ImportError:
    _SM_OK = False
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')
warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')
CHART_DPI = 150
matplotlib.rcParams['figure.dpi'] = CHART_DPI
print(f'pandas {pd.__version__}  |  numpy {np.__version__}  |  rapidfuzz: {_RAPIDFUZZ_OK}')

pandas 2.3.3  |  numpy 2.3.4  |  rapidfuzz: True


## Cell 3.1 — RUN WINDOW, PATHS & THRESHOLDS *(LIFTED VERBATIM from tech_workload v1.35.0 — re-sync if the main notebook changes)*

In [3]:
RUN_DATE     = datetime.now().strftime('%Y-%m-%d')
FILTER_START = '2025-01-01'
AS_OF_DATE   = datetime.now().date() - timedelta(days=1)
FILTER_END   = AS_OF_DATE.strftime('%Y-%m-%d')
print(f'AS_OF_DATE (last full day of data): {AS_OF_DATE.isoformat()}')
EXCLUDE_CURRENT_MONTH = False
if EXCLUDE_CURRENT_MONTH:
    _last = datetime.now().replace(day=1) - relativedelta(days=1)
    FILTER_END = _last.strftime('%Y-%m-%d')

USER_ROOT = '/./Users/aenew'
#USER_ROOT = '/./Users/AlexNewton'
REPORT_ROOT = f'{USER_ROOT}/OneDrive - DME Express/Reports/TechWorkload'
ENV_FILE = f'{USER_ROOT}/OneDrive - DME Express/Documents/IT/Python/MSAIKey.env'  # _LEGACY v1.35.0: login now via Key Vault (Cell 4); delete with the commented .env login after 2026-Q4
DRIVER_NAME = 'ODBC Driver 18 for SQL Server'
SERVER_NAME = 'tcp:dmeexpress.database.windows.net,1433'
DATABASE_NAME = 'DMEEXPRESS'

FUZZY_EDIT_DIST  = 1
OUTLIER_Z_THRESH = 2.5
DATA_FRESHNESS_THRESHOLD = 0.25

MIN_HOURS_MONTH  = 4.0

# ── v1.33.0 (N1): Overtime configuration ────────────────────────────────────
# FLSA-style inference from PLC daily hours: any hours over OT_WEEKLY_THRESHOLD
# in a Sun–Sat workweek count as overtime. Computed on RAW per-person hours
# (before any warehouse pro-rating) because the threshold applies to the
# employee, not the warehouse. CAVEAT: if PLC 'Hours' includes PTO/holiday pay,
# OT is overstated (PTO does not count toward the 40-hr threshold). Confirm the
# feed with payroll; if an earnings-code column exists, prefer it over inference.
OT_WEEKLY_THRESHOLD = 40.0   # hours per workweek before OT begins
MIN_HOURS_FOR_OT_RATE = 80.0 # total-hours floor before a tech appears in OT-rate rankings
MIN_ACTIVE_DAYS_RANK  = 30   # active-day floor for tech top/bottom ranking (M1)
PCT_DEPT = 'Patient Care Technician'


AS_OF_DATE (last full day of data): 2026-08-13


## Cell 3.2 — ROLE CLASSIFICATION & INCLUDED REASONS *(LIFTED VERBATIM from tech_workload v1.35.0 — re-sync if the main notebook changes)*

In [4]:
# Role classification vocabularies: a matched employee whose title/dept hits the
# FIELD_TECH sets is countable workload; INTERNAL_OPS hits (dispatchers, CSRs) are
# excluded from productivity and reported separately. INCLUDED_REASONS is the
# locked ticket-reason whitelist — SQL filters on it, so adding a reason code
# upstream requires adding it here or those tickets silently never arrive.
FIELD_TECH_TITLES = {'patient care technician','service technician','lead technician','lead tech',
                     'warehouse technician','warehouse tech','warehouse manager','site manager','area manager',}
FIELD_TECH_DEPTS = {'field operations','warehouse'}
INTERNAL_OPS_TITLES = {'customer service','csr','dispatcher','dispatch','routing','call center','intake','scheduler','scheduling'}
INTERNAL_OPS_DEPTS = {'customer service','dispatch','routing','call center','intake','scheduling'}

INCLUDED_REASONS = [
    'Priority 1 - Hospital Discharge (D)',
    'Priority 1 - Respiratory Distress (D)',
    'Priority 1 - Respiratory Service/Exchange (D)',
    'Priority 1 - Respiratory Service/Exchange (P)',
    'Priority 1 - Respiratory Service/Exchange (S)',
    'Priority 1 - Service Correction (D)',
    'Priority 1 - Service Correction (P)',
    'Priority 2 - Exchange (S)',
    'Priority 2 - New Admit (D)',
    'Priority 2 - Respiratory Equipment (D)',
    'Priority 2 - Service/Exchange (D)',
    'Priority 2 - Service/Exchange (P)',
    'Priority 2 - Service/Repair (S)',
    'Priority 2 - Swap Out (Equipment Provider)',
    'Priority 3 - Additional Equipment (D)',
    'Priority 3 - Change Address (D)',
    'Priority 3 - Change Address (P)',
    'Priority 3 - Customer/Patient Request (P)',
    'Priority 3 - Exchange (S)',
    'Priority 3 - Inservice (S)',
    'Priority 3 - Live Discharge (P)',
    'Priority 3 - O2 Refill (D)',
    'Priority 3 - O2 Refill (P)',
    'Priority 3 - Patient Expired (P)',
    'Priority 3 - Respite Stay (D)',
    'Priority 3 - Respite Stay (P)',
    'Priority 3 - Service/Exchange (D)',
    'Priority 3 - Service/Exchange (P)',
    'Priority 3 - Swap Out (Equipment Provider)',
    'Split Order',
]


## Cell 3.3 — GEOGRAPHY: METROS, PALETTE, OUTPUT FOLDER *(LIFTED VERBATIM from tech_workload v1.35.0 — re-sync if the main notebook changes)*

In [5]:
STATE_PREFIX_LEN = 2  # _DEPRECATED v1.34.1: prefix fallback removed in v1.26; retained one quarter, delete after 2026-Q4
METRO_GROUPS = {
    'DFW':         ['irving','garland','fort worth','txs garland'],
    'Houston':     ['houston','league city','south houston'],
    'San Antonio': ['san antonio'],
}

PALETTE = {
    'attributed'   : '#1f77b4',
    'dark_unattrib': '#d62728',
    'unmatched'    : '#ff7f0e',
    'blank'        : '#9467bd',
    'redelivery'   : '#e377c2',
    'internal_ops' : '#8c564b',
    'company_avg'  : '#000000',
}

OUT_DIR = os.path.join(REPORT_ROOT, RUN_DATE)
os.makedirs(OUT_DIR, exist_ok=True)
print(f'Output: {OUT_DIR}')
print(f'Window: {FILTER_START} -> {FILTER_END}')
print(f'Metro groups: {list(METRO_GROUPS.keys())}')

# ─────────────────────────────────────────────────────────────────────────────
# WAREHOUSE_STATE_OVERRIDES
# Manual state mapping for warehouses where SERP_WAREHOUSES.[State Province]
# is NULL AND no sibling warehouse (same city, different region prefix) has a
# populated state. The resolution pipeline in Cell 9 tries (1) master,
# (2) sibling-city lookup from df_hier, (3) this override map, (4) 'Unknown'.
# Add new entries here as new orphan warehouses appear.
# ─────────────────────────────────────────────────────────────────────────────


Output: /./Users/aenew/OneDrive - DME Express/Reports/TechWorkload\2026-08-14
Window: 2025-01-01 -> 2026-08-13
Metro groups: ['DFW', 'Houston', 'San Antonio']


## Cell 3.4 — WAREHOUSE → STATE OVERRIDES *(LIFTED VERBATIM from tech_workload v1.35.0 — re-sync if the main notebook changes)*

In [6]:
# Manual state map for warehouses whose SERP_WAREHOUSES.[State Province] is NULL
# and that have no sibling-city row to borrow from. Layer 3 of the resolution
# pipeline in Cell 9.4 — add new orphan warehouses here as they appear.
WAREHOUSE_STATE_OVERRIDES = {
    'Distribution Center - Alabama':   'AL',
    'Distribution Center - Louisiana': 'LA',
    'R03 Hot Springs':       'AR',
    'R04 Camden':            'AR',
    'R04 Fort Smith':        'AR',
    'R05 Natchez Storage':   'MS',
    'R05 Tuscaloosa':        'AL',
    'R08 Alexander City':    'AL',
    'R09 Calhoun':           'GA',
    'R10 Hunt Valley':       'MD',
    'R11 Columbia - SC':     'SC',
    'R12 Garland':           'TX',
    'R13 Akron':             'OH',
    'R14 Hendersonville':    'TN',
    'R14 Jackson, TN':       'TN',
    'R15 Chantilly - VA':    'VA',
    'R15 Fredericksburg':    'VA',
    'R16 H3S':               'TX',
    'R16 Houston':           'TX',
    'RNW Austin':            'TX',
    'RNW Garland':           'TX',
}
print(f'WAREHOUSE_STATE_OVERRIDES: {len(WAREHOUSE_STATE_OVERRIDES)} entries')

WAREHOUSE_STATE_OVERRIDES: 21 entries


## Cell 3.5 — DASHBOARD CONFIGURATION (new)

In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# DASHBOARD CONFIGURATION (v1.0.0)
#
# This notebook is SELF-CONTAINED by explicit decision (2026-08-14): it re-queries
# Azure SQL and duplicates the v1.35.0 matching machinery. Every duplicated cell
# is marked "LIFTED VERBATIM from v1.35.0" — when the main notebook's matching or
# extraction logic changes, those cells must be re-synced or the two reports will
# quietly diverge. That is the accepted cost of standalone operation.
# ─────────────────────────────────────────────────────────────────────────────
DASH_TOP_N          = 5      # top/bottom technicians listed per grouping
DASH_MIN_ACTIVE_DAYS = 30    # active-weekday floor before a tech is rankable (matches MIN_ACTIVE_DAYS_RANK)
DASH_ROLL_WEEKS     = 4      # rolling window on weekly productivity

# ── STOCK OUTS — NOT YET CONFIGURED ──────────────────────────────────────────
# Awaiting the source table/report name (owner to provide). When known, set:
#   STOCKOUT_TABLE   = 'dbo.[<table name>]'
#   STOCKOUT_COLS    = {'warehouse': '<warehouse col>', 'date': '<event date col>',
#                       'product': '<product col>', 'qty': '<qty col or None>'}
# Until then every stock-out panel/sheet is skipped with a visible notice —
# nothing is proxied or fabricated in its place.
STOCKOUT_TABLE = None
STOCKOUT_COLS  = {}

DASH_PDF_NAME  = f'OpsDashboard_Pages_{RUN_DATE}.pdf'
DASH_XLSX_NAME = f'OpsDashboard_Report_{RUN_DATE}.xlsx'
print(f'Dashboard config loaded. Stock-outs configured: {STOCKOUT_TABLE is not None}')


Dashboard config loaded. Stock-outs configured: False


## Cell 4.1 — SECURE CONNECTION: AZURE KEY VAULT *(LIFTED VERBATIM from tech_workload v1.35.0 — re-sync if the main notebook changes)*

In [8]:
# load_dotenv(ENV_FILE)
# sql_conn = odbc.connect(
#    f'DRIVER={{{DRIVER_NAME}}};SERVER={SERVER_NAME};DATABASE={DATABASE_NAME};'
#    f'UID={os.getenv("SQL_USERNAME")};PWD={os.getenv("SQL_PASSWORD")};'
#    f'Encrypt=yes;TrustServerCertificate=no;Connection Timeout=30;'
#)

KEY_VAULT_URI   = 'https://dmee-keyvault.vault.azure.net/'
HOSTNAME_SECRET = 'DataWarehouseHostname'
SQL_USERNAME    = 'anewton-ro'   # secret name is the login; its value is the password

kv = SecretClient(vault_url=KEY_VAULT_URI, credential=DefaultAzureCredential())
SERVER_NAME  = kv.get_secret(HOSTNAME_SECRET).value
SQL_PASSWORD = kv.get_secret(SQL_USERNAME).value
print('Secrets loaded from Key Vault.')

sql_conn = odbc.connect(
    f'DRIVER={{{DRIVER_NAME}}};SERVER={SERVER_NAME};DATABASE={DATABASE_NAME};'
    f'UID={SQL_USERNAME};PWD={SQL_PASSWORD};'
    f'Encrypt=yes;TrustServerCertificate=no;Connection Timeout=30;'
)

print('DB connection established.')


Secrets loaded from Key Vault.
DB connection established.


## Cell 4.2 — QUERY RUNNER & UTILITIES *(LIFTED VERBATIM from tech_workload v1.35.0 — re-sync if the main notebook changes)*

In [9]:
def run_query(query, label='', verbose=False):
    if verbose: print(f'Query: {label}\n{query}')
    t0 = time.time()
    cur = sql_conn.cursor(); cur.execute(query)
    rows = cur.fetchall()
    cols = [c[0].lower() for c in cur.description]
    df   = pd.DataFrame(rows, columns=cols)
    if label: print(f'  {label}: {len(df):,} rows  ({time.time()-t0:.1f}s)')
    return df

def clean_numbers(val):
    # v1.33.0 (L2): preserve the negative sign — the old regex stripped '-', silently
    # flipping negative adjustments positive. Also avoid lstrip('0') mangling.
    if val is None: return np.nan
    s = str(val).strip()
    neg = s.startswith('-') or (s.startswith('(') and s.endswith(')'))  # (123) = accounting negative
    c = re.sub(r'[^0-9.]', '', s)
    if not c or c == '.': return np.nan
    try:
        v = float(c)
    except ValueError:
        return np.nan
    return -v if neg else v

def save_fig(fig, name):
    fig.savefig(os.path.join(OUT_DIR, f'{name}_{RUN_DATE}.png'), bbox_inches='tight', dpi=CHART_DPI)

def _strip_wh_prefix(name):
    """Strip leading region token (R##, RNW) or 'Distribution Center -' to expose
    the underlying city/location. Used by resolve_state_series() to find sibling
    warehouses for the same physical location.
    Examples:
        'R02 Lake Charles'         -> 'Lake Charles'
        'RNW Del Rio'              -> 'Del Rio'
        'Distribution Center - LA' -> 'LA'
    """
    import re as _re
    if not isinstance(name, str): return ''
    s = name.strip()
    s = _re.sub(r'^R\d{2}\s+', '', s)
    s = _re.sub(r'^RNW\s+', '', s, flags=_re.IGNORECASE)
    s = _re.sub(r'^Distribution Center\s*-\s*', '', s, flags=_re.IGNORECASE)
    return s.strip()

def assign_metro(wh_name, metro_groups=METRO_GROUPS):
    """Assign a warehouse to a metro group by substring match (case-insensitive).
    Returns the metro name, or None if no match. First match wins.
    """
    if not wh_name or not isinstance(wh_name, str): return None
    wh_lc = wh_name.lower().strip()
    for metro, patterns in metro_groups.items():
        if any(p in wh_lc for p in patterns):
            return metro
    return None

## Cell 5 — LEVENSHTEIN DISTANCE *(LIFTED VERBATIM from tech_workload v1.35.0 — re-sync if the main notebook changes)*

In [10]:
# Damerau–Levenshtein edit distance for fuzzy first-name matching (Pass 4).
# rapidfuzz (C++) when installed; otherwise a pure-Python fallback with the same
# semantics, including adjacent-transposition ('jhon'→'john' = distance 1).
if _RAPIDFUZZ_OK:
    def levenshtein(a, b): return _DL.distance(a, b)
else:
    def levenshtein(a, b):
        if a==b: return 0
        if not a: return len(b)
        if not b: return len(a)
        la,lb=len(a),len(b)
        d=[[0]*(lb+1) for _ in range(la+1)]
        for i in range(la+1): d[i][0]=i
        for j in range(lb+1): d[0][j]=j
        for i in range(1,la+1):
            for j in range(1,lb+1):
                cost=0 if a[i-1]==b[j-1] else 1
                d[i][j]=min(d[i-1][j]+1,d[i][j-1]+1,d[i-1][j-1]+cost)
                if i>1 and j>1 and a[i-1]==b[j-2] and a[i-2]==b[j-1]: d[i][j]=min(d[i][j],d[i-2][j-2]+cost)
        return d[la][lb]
print('levenshtein() ready.')

levenshtein() ready.


## Cell 6 — NICKNAME DICTIONARY *(LIFTED VERBATIM from tech_workload v1.35.0 — re-sync if the main notebook changes)*

In [11]:
# Bidirectional nickname dictionary (Pass 3): 'mike'→Michael AND Michael→'mike'.
# Includes Spanish hypocorisms (pepe/chuy/beto...) for the field workforce.
# Retained as-is by explicit decision — do not prune without a match-rate test.
NICKNAME_TO_CANONICAL_RAW = [
    ('chris',['Christopher','Christian','Christina','Christine']),('kris',['Christopher','Kristopher']),
    ('mike',['Michael']),('mikey',['Michael']),('matt',['Matthew']),('dan',['Daniel']),('danny',['Daniel']),
    ('dave',['David']),('rob',['Robert']),('bob',['Robert']),('bobby',['Robert']),('robbie',['Robert']),
    ('robby',['Robert']),('jim',['James']),('jimmy',['James']),('jamie',['James']),('joe',['Joseph']),
    ('joey',['Joseph']),('tom',['Thomas']),('tommy',['Thomas']),('bill',['William']),('billy',['William']),
    ('will',['William']),('liam',['William']),('rick',['Richard','Ricardo','Frederick']),
    ('ricky',['Richard','Ricardo']),('rich',['Richard']),('steve',['Steven','Stephen']),
    ('ed',['Edward','Eduardo','Edwin']),('eddie',['Edward','Eduardo']),('ted',['Edward','Theodore']),
    ('tony',['Anthony','Antonio']),('nick',['Nicholas','Nicolas']),('pat',['Patrick','Patricia']),
    ('tim',['Timothy']),('timmy',['Timothy']),('jon',['Jonathan','Jonathon']),
    ('johnny',['John','Jonathan']),('jack',['John','Jackson']),('jeff',['Jeffrey','Geoffrey']),
    ('andy',['Andrew','Andres']),('drew',['Andrew']),('ron',['Ronald','Ronaldo']),
    ('ronnie',['Ronald']),('ken',['Kenneth']),('kenny',['Kenneth']),('ben',['Benjamin']),
    ('benny',['Benjamin','Benito']),('greg',['Gregory']),('sam',['Samuel','Samantha']),
    ('josh',['Joshua']),('alex',['Alexander','Alejandro','Alexandra']),
    ('nate',['Nathaniel','Nathan']),('zach',['Zachary','Zachariah']),('zack',['Zachary']),
    ('mitch',['Mitchell']),('ray',['Raymond','Raymundo']),('larry',['Lawrence','Lorenzo']),
    ('terry',['Terrence','Terrell']),('jerry',['Gerald','Jeremiah','Jerome']),
    ('chuck',['Charles']),('charlie',['Charles']),('fred',['Frederick','Fredrick','Alfredo']),
    ('hank',['Henry']),('harry',['Henry','Harold','Harrison']),('phil',['Philip','Phillip']),
    ('wes',['Wesley']),('vince',['Vincent']),('vinny',['Vincent','Vincenzo']),
    ('abe',['Abraham']),('gabe',['Gabriel']),('len',['Leonard']),('walt',['Walter']),
    ('doug',['Douglas']),('al',['Albert','Alan','Alfonso']),('bert',['Albert','Robert','Herbert']),
    ('don',['Donald']),('gene',['Eugene']),('manny',['Manuel','Emmanuel']),('marty',['Martin']),
    ('art',['Arthur']),('curt',['Curtis']),('ernie',['Ernest','Ernesto']),
    ('frank',['Franklin','Francisco','Francis']),('frankie',['Franklin','Francisco','Frank']),
    ('jr',['Junior']),('lupe',['Guadalupe']),('max',['Maximilian','Maxwell','Maximo']),
    ('reggie',['Reginald']),('rudy',['Rudolph','Rodolfo']),
    ('ty',['Tyler','Tyrone','Tyson']),('vic',['Victor']),
    ('liz',['Elizabeth']),('beth',['Elizabeth']),('lisa',['Elizabeth']),
    ('kate',['Katherine','Kathryn','Kaitlyn']),('kathy',['Katherine','Kathryn']),
    ('katie',['Katherine']),('sue',['Susan','Suzanne']),('susie',['Susan']),
    ('jen',['Jennifer']),('jenny',['Jennifer']),('jenn',['Jennifer']),('amy',['Amelia','Amy']),
    ('meg',['Megan','Margaret']),('maggie',['Margaret']),('pam',['Pamela']),('barb',['Barbara']),
    ('deb',['Deborah','Debra']),('debbie',['Deborah','Debra']),('carol',['Caroline','Carolyn']),
    ('tina',['Christina']),('sandy',['Sandra','Alexandra']),('cindy',['Cynthia']),
    ('angie',['Angela','Angelica']),('ang',['Angela']),('steph',['Stephanie']),
    ('stacy',['Stacey','Stacy']),('nikki',['Nicole','Nichole']),('mia',['Maria']),
    ('maria',['Maria','Marie']),('anna',['Annette','Annalisa']),
    ('nicky',['Nichole','Nicole']),('joanie',['Joan']),('diana',['Diane']),('dani',['Danielle']),
    ('trish',['Patricia']),('patty',['Patricia']),('bri',['Brianna','Brittany']),
    ('brit',['Brittany','Britney']),('chrissy',['Christina','Christine']),('vero',['Veronica']),
    ('cass',['Cassandra']),('dot',['Dorothy']),('bev',['Beverly']),
    ('mel',['Melanie','Melissa','Melinda']),('missy',['Melissa']),('mindy',['Melinda']),
    ('pepe',['Jose']),('chuy',['Jesus']),('nacho',['Ignacio']),('chava',['Salvador']),
    ('memo',['Guillermo']),('beto',['Roberto','Alberto']),('pancho',['Francisco']),
    ('paco',['Francisco']),('chela',['Graciela']),('chucho',['Jesus']),('vale',['Valeria']),
    ('lalo',['Eduardo']),('pipe',['Felipe']),('nico',['Nicolas']),
]
NICKNAME_TO_CANONICAL = {}
for nick,canonicals in NICKNAME_TO_CANONICAL_RAW:
    if nick not in NICKNAME_TO_CANONICAL: NICKNAME_TO_CANONICAL[nick]=[]
    for c in canonicals:
        if c not in NICKNAME_TO_CANONICAL[nick]: NICKNAME_TO_CANONICAL[nick].append(c)
CANONICAL_TO_NICKNAMES = defaultdict(list)
for nick,canonicals in NICKNAME_TO_CANONICAL.items():
    for canon in canonicals: CANONICAL_TO_NICKNAMES[canon.lower()].append(nick)

def standardize_first_name(raw):
    if not raw or not isinstance(raw,str): return []
    raw_lc=raw.lower().strip(); seen,results=set(),[raw]; seen.add(raw_lc)
    for cand in NICKNAME_TO_CANONICAL.get(raw_lc,[]):
        if cand.lower() not in seen: results.append(cand); seen.add(cand.lower())
    for nick in CANONICAL_TO_NICKNAMES.get(raw_lc,[]):
        if nick.lower() not in seen: results.append(nick.title()); seen.add(nick.lower())
    return results
print(f'Nickname map: {len(NICKNAME_TO_CANONICAL)} entries')

Nickname map: 152 entries


## Cell 7.1 — TRANSACTIONS EXTRACT + H3 PROBE *(LIFTED VERBATIM from tech_workload v1.35.0 — re-sync if the main notebook changes)*

In [12]:
print(f'Extracting data ({FILTER_START} to {FILTER_END})...')
t0 = time.time()
_reasons_sql = ',\n      '.join(f"'{r}'" for r in INCLUDED_REASONS)

df_tx = run_query(f"""
SELECT TRIM(TX.Order_Num) AS order_num, TRIM(TX.Record_ID) AS record_id,
    TRIM(TX.Tech_Warehouse) AS tech_warehouse,
    TRIM(ISNULL(TX.TechFirstName,'')) AS techfirstname,
    TRIM(ISNULL(TX.TechLastName,''))  AS techlastname,
    TX.Reason AS reason,
    TRY_CONVERT(DATE, TX.Completed_Date) AS completed_date
FROM dbo.[SERP TRANSACTIONS] AS TX WITH (NOLOCK)
WHERE TX.Tech_Warehouse NOT LIKE 'Z%' AND TX.[Status] <> 'Canceled'
  AND TX.Order_Num <> ''
  -- v1.33.0 (H3): filter on the CONVERTED date, matching the SELECT. The raw-string
  -- compare could (a) drop the entire last day if values carry a time component
  -- ('2026-07-30 14:22' > '2026-07-30' as strings) and (b) admit malformed strings
  -- that TRY_CONVERT NULLs in the SELECT, creating NaT rows downstream.
  AND TRY_CONVERT(DATE, TX.Completed_Date) >= '{FILTER_START}'
  AND TRY_CONVERT(DATE, TX.Completed_Date) <= '{FILTER_END}'
  AND TX.Reason IN ({_reasons_sql})
OPTION (RECOMPILE, MAXDOP 4)
""", 'Transactions')
display(df_tx.head(3))

# v1.34.1 — H3 probe. The v1.33.0 WHERE clause (TRY_CONVERT) excludes rows whose
# Completed_Date cannot be parsed as DATE, and the Cell 8 NaT guard catches any
# that slip through. Neither reports how many rows the WHERE silently excluded —
# this probe does, every run, so a feed-format regression can't hide.
_h3 = run_query("""
SELECT COUNT(*) AS bad_rows FROM dbo.[SERP TRANSACTIONS] WITH (NOLOCK)
WHERE TRY_CONVERT(DATE, Completed_Date) IS NULL
  AND Completed_Date IS NOT NULL AND LTRIM(RTRIM(Completed_Date)) <> ''
  AND [Status] <> 'Canceled' AND Order_Num <> ''
""", 'H3 unparseable Completed_Date probe')
_h3_n = int(_h3.iloc[0, 0]) if len(_h3) else 0
if _h3_n > 0:
    print(f'  *** H3: {_h3_n:,} transaction rows have UNPARSEABLE Completed_Date and are')
    print(f'      EXCLUDED from all analytics. Pull samples and fix the upstream feed.')
else:
    print('  H3 probe: 0 unparseable Completed_Date rows — verification item closed for this run.')


Extracting data (2025-01-01 to 2026-08-13)...
  Transactions: 449,326 rows  (40.8s)


,order_num,record_id,tech_warehouse,techfirstname,techlastname,reason,completed_date
0,3133624,646029,R15 Chesapeake,Yale,Pinnix,Priority 3 - Additional Equipment (D),2026-06-09
1,3133627,647678,R12 Irving,James,Hudson,Priority 2 - Swap Out (Equipment Provider),2026-06-17
2,3133636,644484,R15 T Storage,Lewis,Ball,Priority 3 - Additional Equipment (D),2026-06-08


  H3 unparseable Completed_Date probe: 1 rows  (11.7s)
  H3 probe: 0 unparseable Completed_Date rows — verification item closed for this run.


## Cell 7.2 — EMPLOYEES & WAREHOUSE HIERARCHY *(LIFTED VERBATIM from tech_workload v1.35.0 — re-sync if the main notebook changes)*

In [13]:
df_emp = run_query("""
SELECT count(id) AS emp_duplicate_count,
    TRIM([FirstName]) AS empfirstname, TRIM([LastName]) AS emplastname,
    TRIM([Warehouse Name]) AS [location],
    MAX([Department Name]) AS dept, MAX([Job Title]) AS title,
    MAX([Employee Number]) AS eid, MAX([Username]) AS username
FROM SERP_DME_EMPLOYEES
GROUP BY TRIM([FirstName]),TRIM([LastName]),TRIM([Warehouse Name])
""", 'Employees')

df_hier = run_query("""
SELECT DISTINCT TRIM([Warehouse Name]) AS warehouse, [State Province] as [State],
    LEFT(TRIM([Warehouse Name]),3) AS region, TRIM([Group]) AS vp
FROM SERP_WAREHOUSES WITH (NOLOCK)
""", 'Warehouse hierarchy')


  Employees: 2,003 rows  (0.4s)
  Warehouse hierarchy: 138 rows  (0.0s)


## Cell 7.3 — REDELIVERIES EXTRACT *(LIFTED VERBATIM from tech_workload v1.35.0 — re-sync if the main notebook changes)*

In [14]:
df_redel = run_query(f"""
SELECT TRIM(RD.Orig_Order) AS orig_order_num,
    TRY_CONVERT(DATE,RD.Completion_DateTime) AS rd_date,
    RD.Completion_DateTime AS rd_datetime_raw,
    TRIM(RD.Products) AS rd_products,
    TRIM(ISNULL(SE.[FirstName],'')) AS techfirstname,
    TRIM(ISNULL(SE.[LastName],''))  AS techlastname,
    TRIM(ISNULL(RD.Tech_Warehouse,'')) AS tech_warehouse
FROM dbo.[Re-Delivery Report] RD
LEFT JOIN dbo.SERP_DME_EMPLOYEES SE ON SE.Username=RD.Tech
WHERE (TRY_CONVERT(DATE,RD.Completion_DateTime) BETWEEN '{FILTER_START}' AND '{FILTER_END}')
   OR (TRY_CONVERT(DATE,RD.Completion_DateTime) IS NULL
       AND RD.Completion_DateTime LIKE '202_-%')  -- keep parse-fail rows that look in-range
""", 'Redeliveries')
if 'tech_warehouse' not in df_redel.columns: df_redel['tech_warehouse']=''


  Redeliveries: 187,737 rows  (3.8s)


## Cell 7.4 — LOST EQUIPMENT & INVENTORY EXTRACT *(LIFTED VERBATIM from tech_workload v1.35.0 — re-sync if the main notebook changes)*

In [15]:
df_lost_raw = run_query(f"""
SELECT ATI.Asset_Tag AS asset_tag,
    NULLIF(TRIM(CAST(ATI.Bill_to_ID AS VARCHAR(20))),'0') AS bill_to_id,
    MP.Product_Name AS product_name, WH.[Warehouse Name] AS tech_warehouse,
    LEFT(TRIM(WH.[Warehouse Name]),3) AS region, TRIM(WH.[Group]) AS vp,
    WH.[State Province] AS state,  -- v1.26.0: state from master, not prefix
    ATI.Lost_Date AS lost_date_raw, MP.Unit_Cost_Last_Price AS lost_cost_last_price
FROM SERP_ACTIVE_TAGGED_INV ATI WITH (NOLOCK)
JOIN SERP_WAREHOUSES WH WITH (NOLOCK) ON WH.ID=ATI.Warehouse_ID
JOIN SERP_MASTER_PRODUCTS MP WITH (NOLOCK) ON MP.ID=ATI.Master_ID
WHERE ATI.Lost IS NOT NULL AND WH.[Warehouse Name] NOT LIKE 'Z%'
  AND ATI.Lost_Date >= '{FILTER_START}'
  AND ATI.Lost_Date <= '{FILTER_END}'  -- v1.30.4
""", 'Lost equipment')

df_inventory_total = run_query("""
SELECT TRIM(WH.[Warehouse Name]) AS tech_warehouse, 
    COUNT_BIG(ATI.Asset_Tag) AS total_inventory_count,
    SUM(CONVERT(float, COALESCE(ATI.Unit_Cost, MP.Unit_Cost_Last_Price))) AS total_inventory_amount
FROM SERP_ACTIVE_TAGGED_INV ATI WITH (NOLOCK)
JOIN SERP_WAREHOUSES WH WITH (NOLOCK) ON WH.ID=ATI.Warehouse_ID
LEFT OUTER JOIN SERP_MASTER_PRODUCTS MP ON MP.ID=ATI.Master_ID
WHERE WH.[Warehouse Name] NOT LIKE 'Z%' AND MP.Active='Yes' AND MP.Asset_Tag_Not_Required = 'No'
GROUP BY TRIM(WH.[Warehouse Name])
""", 'Inventory totals')
df_inventory_total['total_inventory_count'] = pd.to_numeric(df_inventory_total['total_inventory_count'], errors='coerce').fillna(0).astype(int)
df_inventory_total['total_inventory_amount'] = pd.to_numeric(df_inventory_total['total_inventory_amount'], errors='coerce').fillna(0).astype(float)

df_master = run_query("SELECT ID AS masterid, TRIM(Product_Name) AS product_name FROM dbo.SERP_MASTER_PRODUCTS", 'Master products')
print(f'All queries complete in {time.time()-t0:.1f}s')

  Lost equipment: 0 rows  (0.1s)
  Inventory totals: 85 rows  (0.2s)
  Master products: 1,117 rows  (0.0s)
All queries complete in 57.1s


## Cell 7.5 — PATIENT CENSUS (APC/ADC) *(LIFTED VERBATIM from tech_workload v1.35.0 — re-sync if the main notebook changes)*

Needed because the lifted lost-equipment prep (Cell 12) merges monthly ADC as a
census denominator for lost-rate context.


In [16]:
df_apc_snapshot_query = f"""
SELECT TOP 1 [date] FROM SERP_APC_DAILY
WHERE [date] <= '{FILTER_END}'
ORDER BY [date] DESC
"""
_apc_snap_df = run_query(df_apc_snapshot_query, 'APC snapshot-date probe')
if len(_apc_snap_df)==0 or pd.isna(_apc_snap_df.iloc[0,0]):
    raise RuntimeError(f'APC: no rows on or before {FILTER_END}. Cannot compute snapshot.')
_apc_snap_date = pd.to_datetime(_apc_snap_df.iloc[0,0]).date()
_apc_lag_days = (AS_OF_DATE - _apc_snap_date).days
print(f'  APC snapshot date: {_apc_snap_date.isoformat()} (lag: {_apc_lag_days} day(s) behind AS_OF_DATE)')
if _apc_lag_days > 7:
    print(f'  *** WARNING: APC feed is >7 days stale. Lost-cost-per-ADC metrics may be unreliable.')
df_apc = run_query(f"""
SELECT TRIM(APC.warehourse) AS warehouse, SUM(APC.total) AS apc
FROM SERP_APC_DAILY AS APC WITH (NOLOCK)
WHERE APC.[date] = '{_apc_snap_date.isoformat()}'
  AND APC.warehourse NOT LIKE 'Z%'
  AND APC.customer NOT LIKE '(F)%' AND APC.customer NOT LIKE '(IPU)%'
  AND APC.customer NOT LIKE '%Contract Test%'
GROUP BY TRIM(APC.warehourse)
""", 'APC')
df_apc['apc'] = df_apc['apc'].apply(clean_numbers)

df_adc = run_query(f"""
WITH daily AS (SELECT [date],warehourse,SUM(total) AS APC FROM SERP_APC_DAILY
    WHERE [date]>='{FILTER_START}' AND [date]<='{FILTER_END}' AND warehourse NOT LIKE 'Z%' GROUP BY [date],warehourse)  -- v1.30.4
SELECT TRIM(warehourse) AS warehouse,AVG(APC) AS ADC,SUM(APC) AS pt_days,
    YEAR([date]) AS yr,MONTH([date]) AS mo
FROM daily GROUP BY YEAR([date]),MONTH([date]),warehourse ORDER BY yr,mo,warehourse
""", 'ADC')
df_adc[['yr','mo']] = df_adc[['yr','mo']].astype(int)
df_adc['adc']     = df_adc['adc'].apply(clean_numbers)
df_adc['pt_days'] = df_adc['pt_days'].apply(clean_numbers)


  APC snapshot-date probe: 1 rows  (0.2s)
  APC snapshot date: 2026-08-13 (lag: 0 day(s) behind AS_OF_DATE)
  APC: 63 rows  (0.1s)
  ADC: 1,219 rows  (0.3s)


## Cell 8.1 — DATES, NaT GUARD, SCHEDULE PERIOD & METRO *(LIFTED VERBATIM from tech_workload v1.35.0 — re-sync if the main notebook changes)*

In [17]:
df_tx['completed_date'] = pd.to_datetime(df_tx['completed_date'],errors='coerce')
# v1.33.0 (H3): NaT guard. With the WHERE now on TRY_CONVERT this should be zero;
# if it isn't, rows were admitted that can't be dated and would silently form
# NaN month groups (or crash later int casts). Fail loudly, drop, and report.
_n_nat = int(df_tx['completed_date'].isna().sum())
if _n_nat > 0:
    print(f'*** WARNING: {_n_nat:,} transaction rows have unparseable Completed_Date — DROPPED.')
    df_tx = df_tx[df_tx['completed_date'].notna()].copy()
df_tx['delivery_year']  = df_tx['completed_date'].dt.year
df_tx['delivery_month'] = df_tx['completed_date'].dt.month
df_tx['month_date']     = df_tx['completed_date'].values.astype('datetime64[M]')
_dow = df_tx['completed_date'].dt.dayofweek
df_tx['schedule_period'] = np.select([_dow==5,_dow==6],['Saturday','Sunday'],default='Weekday')
df_tx['metro'] = df_tx['tech_warehouse'].apply(assign_metro)


## Cell 8.2 — REASON CLASSIFICATION & TICKET TYPE *(LIFTED VERBATIM from tech_workload v1.35.0 — re-sync if the main notebook changes)*

In [18]:
_REASON_MAP = [
    (r'priority 1','Urgent',1),(r'priority 2 - exchange','Exchange/Service',2),
    (r'priority 2 - new admit','New Admission',2),(r'priority 2 - respiratory','Respiratory',2),
    (r'priority 2','Exchange/Service',2),(r'priority 3 - additional','Additional Equip',3),
    (r'priority 3 - change','Admin/Change',3),(r'priority 3 - customer','Patient Request',3),
    (r'priority 3 - live disc','Live Discharge',3),(r'priority 3 - o2','O2 Refill',3),
    (r'priority 3 - patient exp','Patient Expired',3),(r'priority 3 - respite','Respite',3),
    (r'priority 3','Routine P3',3),(r'split order','Split Order',4),
]
def _classify(reason):
    if not reason or not isinstance(reason,str): return ('Other',5)
    r=reason.lower()
    for pat,cat,pri in _REASON_MAP:
        if re.search(pat,r): return (cat,pri)
    return ('Other',5)
def _ticket_type(reason):
    if not reason or not isinstance(reason,str): return 'Other'
    if reason.strip().upper()=='SPLIT ORDER': return 'Split'
    m=re.search(r'\(([DPS])\)\s*$',reason.strip())
    return {'D':'Delivery','P':'Pickup','S':'Service'}[m.group(1)] if m else 'Other'

_unique_reasons    = df_tx['reason'].dropna().unique()
_reason_cat_lookup = {r:_classify(r)[0] for r in _unique_reasons}
_reason_pri_lookup = {r:_classify(r)[1] for r in _unique_reasons}
_type_lookup       = {r:_ticket_type(r) for r in _unique_reasons}
df_tx['reason_category'] = df_tx['reason'].map(_reason_cat_lookup).fillna('Other')
df_tx['priority_level']  = df_tx['reason'].map(_reason_pri_lookup).fillna(5).astype(int)
df_tx['ticket_type']     = df_tx['reason'].map(_type_lookup).fillna('Other')

print(f'Rows: {len(df_tx):,}')
print(df_tx['schedule_period'].value_counts().to_string())
# State diagnostic moved to Cell 9 (v1.26.1) — state isn't merged in yet here.
print('\nMetro distribution:')
print(df_tx['metro'].value_counts(dropna=False).to_string())


Rows: 449,326
schedule_period
Weekday     419658
Saturday     19470
Sunday       10198

Metro distribution:
metro
None           355126
San Antonio     35112
Houston         32089
DFW             26999


## Cell 9.1 — EMPLOYEE INDEXES & ROLE CLASSIFIERS *(LIFTED VERBATIM from tech_workload v1.35.0 — re-sync if the main notebook changes)*

In [19]:
# Name normalizer used by every matching pass: strip apostrophes, periods,
# spaces, hyphens; lowercase. "O'Brien" == "OBrien" == "o brien".
# MANUAL_NAME_CORRECTIONS (Pass 0) fixes known-bad spellings before any lookup.
def _clean(s):
    if not s or not isinstance(s,str): return ''
    return re.sub(r"['\.\s\-]",'',s).lower()

MANUAL_NAME_CORRECTIONS = {
    ('damein','combs'):('Damien','Combs'),
    ('damien','combs'):('Damien','Combs'),
}

emp_idx={}; _emp_by_last=defaultdict(list)
for row in df_emp.itertuples(index=False):
    fn,ln=_clean(row.empfirstname),_clean(row.emplastname)
    if fn and ln: emp_idx[(fn,ln)]=row; _emp_by_last[ln].append((fn,row))
print(f'Employee index: {len(emp_idx):,}')

def _classify_role(dept_raw,title_raw):
    d=(dept_raw or '').lower().strip(); t=(title_raw or '').lower().strip()
    if any(x in t for x in FIELD_TECH_TITLES) or any(x in d for x in FIELD_TECH_DEPTS): return 'field_tech'
    if any(x in t for x in INTERNAL_OPS_TITLES) or any(x in d for x in INTERNAL_OPS_DEPTS): return 'internal_ops'
    return 'other'

_dispatcher_last_names=set()
for row in df_emp.itertuples(index=False):
    if any(d in (row.dept or '').lower() for d in INTERNAL_OPS_DEPTS):
        ln_c=_clean(row.emplastname)
        if ln_c: _dispatcher_last_names.add(ln_c)

_tech_by_wh_first={}; _wh_first_collisions=[]
for row in df_emp.itertuples(index=False):
    if _classify_role(row.dept,row.title)!='field_tech': continue
    wh_c,fn_c=_clean(row.location),_clean(row.empfirstname)
    if not (wh_c and fn_c): continue
    key=(wh_c,fn_c)
    if key in _tech_by_wh_first: _wh_first_collisions.append({'wh':row.location,'first':row.empfirstname})
    else: _tech_by_wh_first[key]=row

_field_tech_by_wh_last={}; _wh_last_collisions={}
for row in df_emp.itertuples(index=False):
    if _classify_role(row.dept,row.title)!='field_tech': continue
    wh_c,ln_c=_clean(row.location),_clean(row.emplastname)
    if not (wh_c and ln_c): continue
    key=(wh_c,ln_c)
    if key in _field_tech_by_wh_last: _wh_last_collisions.setdefault(key,[_field_tech_by_wh_last[key]]).append(row)
    else: _field_tech_by_wh_last[key]=row
for key in list(_wh_last_collisions): _field_tech_by_wh_last.pop(key,None)

_internal_ops_first_names=set(); _internal_ops_last_names=set()
for row in df_emp.itertuples(index=False):
    if _classify_role(row.dept,row.title)=='internal_ops':
        fn_c=_clean(row.empfirstname); ln_c=_clean(row.emplastname)
        if fn_c: _internal_ops_first_names.add(fn_c)
        if ln_c: _internal_ops_last_names.add(ln_c)


Employee index: 1,995


## Cell 9.2 — MATCHER FUNCTION (PASSES 0–5) *(LIFTED VERBATIM from tech_workload v1.35.0 — re-sync if the main notebook changes)*

In [20]:
def _match_name(fn_raw,ln_raw,wh_raw=''):
    """Passes 0-5 name matching. Passes 6-7 run in the caller loop."""
    fn_lc,ln_lc=fn_raw.lower(),ln_raw.lower()
    corrected=MANUAL_NAME_CORRECTIONS.get((fn_lc,ln_lc))
    if corrected: fn_raw,ln_raw=corrected
    fn_c,ln_c=_clean(fn_raw),_clean(ln_raw); wh_c=_clean(wh_raw) if wh_raw else ''
    if (fn_c,ln_c) in emp_idx: return emp_idx[(fn_c,ln_c)],('P0' if corrected else 'P1')
    for cand in standardize_first_name(fn_raw)[1:]:
        key=(_clean(cand),ln_c)
        if key in emp_idx: return emp_idx[key],'P2'
    best_row,best_dist,best_same_wh=None,FUZZY_EDIT_DIST+1,False
    for (emp_fn,emp_row) in _emp_by_last.get(ln_c,[]):
        d=levenshtein(fn_c,emp_fn); same_wh=(wh_c!='' and _clean(getattr(emp_row,'location','') or '')==wh_c)
        if d<best_dist or (d==best_dist and same_wh and not best_same_wh):
            best_row,best_dist,best_same_wh=emp_row,d,same_wh
    if best_row is not None and best_dist<=FUZZY_EDIT_DIST: return best_row,'P3'
    if ln_c in _dispatcher_last_names and wh_c:
        match=_tech_by_wh_first.get((wh_c,fn_c))
        if match is not None: return match,'P4'
    if wh_c and ln_c:
        match=_field_tech_by_wh_last.get((wh_c,ln_c))
        if match is not None: return match,'P5_fallback'
    return None,None


## Cell 9.3 — MATCHING LOOP (PASSES 6–7 OVERRIDES) *(LIFTED VERBATIM from tech_workload v1.35.0 — re-sync if the main notebook changes)*

In [21]:
# THE MATCHING LOOP — one pass over each distinct (first, last, warehouse) name
# combination seen on tickets (not per ticket row: cheaper and idempotent).
# Order of operations per name: blank check → passes 0–5 via _match_name() →
# passes 5b/6/7 internal-ops overrides (a dispatcher name on a ticket usually
# means the dispatcher ENTERED it; if a same-warehouse field tech shares the
# last/first name, re-attribute to the tech) → collision bookkeeping.
_unique_names=df_tx[['techfirstname','techlastname','tech_warehouse']].drop_duplicates()
_match_results=[]
_pass_counts={'P0':0,'P1':0,'P2':0,'P3':0,'P4':0,'P5_fallback':0,'P5_intops_override':0,
              'P6_intops_first_override':0,'P7_intops_last_override':0,
              'ambiguous_collision':0,'unmatched_blank':0,'unmatched':0}

for row in _unique_names.itertuples(index=False):  # each distinct name-warehouse combo, once
    fn,ln,wh=row.techfirstname,row.techlastname,row.tech_warehouse
    is_blank=not fn.strip() and not ln.strip()
    pass_used,matched,collision_candidates=None,None,[]
    if is_blank:
        _pass_counts['unmatched_blank']+=1
    else:
        matched,pass_used=_match_name(fn,ln,wh)
        if matched is not None and _classify_role(matched.dept,matched.title)=='internal_ops' and wh and ln:
            _p5=_field_tech_by_wh_last.get((_clean(wh),_clean(ln)))
            if _p5 is not None: matched,pass_used=_p5,'P5_intops_override'
        if wh and ln and pass_used!='P5_intops_override':
            fn_c,ln_c,wh_c=_clean(fn),_clean(ln),_clean(wh)
            if fn_c in _internal_ops_first_names:
                _p6=_field_tech_by_wh_last.get((wh_c,ln_c))
                _cur_ok=(matched is not None and _classify_role(matched.dept,matched.title)=='field_tech' and _clean(getattr(matched,'location','') or '')==wh_c)
                if _p6 is not None and not _cur_ok: matched,pass_used=_p6,'P6_intops_first_override'
        if wh and fn and pass_used not in ('P5_intops_override','P6_intops_first_override'):
            fn_c7,ln_c7,wh_c7=_clean(fn),_clean(ln),_clean(wh)
            if ln_c7 in _internal_ops_last_names:
                _p7=_tech_by_wh_first.get((wh_c7,fn_c7))
                _cur_ok7=(matched is not None and _classify_role(matched.dept,matched.title)=='field_tech' and _clean(getattr(matched,'location','') or '')==wh_c7)
                if _p7 is not None and not _cur_ok7: matched,pass_used=_p7,'P7_intops_last_override'
        # If still unmatched, record whether it's an AMBIGUOUS collision (two field
        # techs share this last name at this warehouse) — exported for HR review.
        if matched is None and wh and ln:
            key=(_clean(wh),_clean(ln))
            if key in _wh_last_collisions: collision_candidates=_wh_last_collisions[key]; _pass_counts['ambiguous_collision']+=1
        _pass_counts['unmatched' if matched is None else pass_used]=_pass_counts.get('unmatched' if matched is None else pass_used,0)+1
    _rb=_classify_role(matched.dept,matched.title) if matched else None
    _match_results.append({'techfirstname':fn,'techlastname':ln,'tech_warehouse':wh,
        '_is_blank_tech':is_blank,'_is_unmatched':(not is_blank)and(matched is None),
        '_is_ambiguous':(not is_blank)and(matched is None)and len(collision_candidates)>0,
        '_collision_candidates':'; '.join(f'{r.empfirstname} {r.emplastname} ({r.eid})' for r in collision_candidates) if collision_candidates else '',
        '_matched_dept':matched.dept if matched else None,'_matched_title':matched.title if matched else None,
        '_matched_eid':matched.eid if matched else None,'_matched_first':matched.empfirstname if matched else None,
        '_matched_last':matched.emplastname if matched else None,'_matched_location':matched.location if matched else None,
        '_matched_role_bucket':_rb,'_is_internal_ops':_rb=='internal_ops','_pass_used':pass_used})


## Cell 9.4 — MERGE RESULTS, HIERARCHY & STATE RESOLUTION *(LIFTED VERBATIM from tech_workload v1.35.0 — re-sync if the main notebook changes)*

In [22]:
_df_match=pd.DataFrame(_match_results)
df_tx=df_tx.merge(_df_match,on=['techfirstname','techlastname','tech_warehouse'],how='left')
for col in ['_is_blank_tech','_is_unmatched','_is_ambiguous','_is_internal_ops']: df_tx[col]=df_tx[col].fillna(False)
df_tx['_matched_role_bucket']=df_tx['_matched_role_bucket'].fillna('unknown')
df_tx['_unattributed']=df_tx['_is_blank_tech']|df_tx['_is_unmatched']|df_tx['_is_internal_ops']

_intops_override_passes={'P5_intops_override','P6_intops_first_override','P7_intops_last_override'}
_correction_mask=df_tx['_pass_used'].isin(_intops_override_passes)
df_tx['_raw_techfirstname']=df_tx['techfirstname']; df_tx['_raw_techlastname']=df_tx['techlastname']
df_tx['_name_was_corrected']=_correction_mask
df_tx.loc[_correction_mask,'techfirstname']=df_tx.loc[_correction_mask,'_matched_first'].fillna(df_tx.loc[_correction_mask,'techfirstname'])
df_tx.loc[_correction_mask,'techlastname'] =df_tx.loc[_correction_mask,'_matched_last'].fillna(df_tx.loc[_correction_mask,'techlastname'])
df_tx=df_tx.merge(df_hier.rename(columns={'warehouse':'tech_warehouse'}),on='tech_warehouse',how='left')
# Layer 1 — value already present from SERP_WAREHOUSES.[State Province] merge.
# Layer 2 — sibling-city lookup: another warehouse with the same trimmed city
#           name (prefix stripped) that has a populated state. Self-healing
#           because as ops backfills the master, this layer picks it up.
# Layer 3 — WAREHOUSE_STATE_OVERRIDES (curated in Cell 3) for orphans.
# Layer 4 — 'Unknown' with a LOUD warning listing every warehouse so the data
#           team can fix the source. NO prefix fallback (it fabricated R0/R1/DI/RN).

def resolve_state_series(wh_series, current_state_series, hier_df):
    """Vectorized 3-layer state resolution. Returns (new_state_series, audit_dict).
    hier_df: a DataFrame with columns ['warehouse','state'] from df_hier."""
    out = current_state_series.copy()
    missing_mask = out.isna() | (out.astype(str).str.strip()=='')
    audit = {'sibling':[], 'override':[], 'unknown':[]}
    if not missing_mask.any():
        return out, audit

    # Build city->state map from hier_df where state IS populated
    _hier_clean = hier_df[hier_df['state'].notna() & (hier_df['state'].astype(str).str.strip()!='')].copy()
    _hier_clean['_city'] = _hier_clean['warehouse'].apply(_strip_wh_prefix)
    # If a city maps to >1 distinct state in the master, that's ambiguous — drop it
    _city_states = _hier_clean.groupby('_city')['state'].nunique()
    _unambig_cities = set(_city_states[_city_states==1].index)
    _city_to_state = (_hier_clean[_hier_clean['_city'].isin(_unambig_cities)]
                      .drop_duplicates('_city').set_index('_city')['state'].to_dict())

    for idx in out[missing_mask].index:
        wh = wh_series.loc[idx]
        if not isinstance(wh, str) or not wh.strip():
            audit['unknown'].append(wh); out.loc[idx] = 'Unknown'; continue
        # Layer 2: sibling city
        city = _strip_wh_prefix(wh)
        if city in _city_to_state:
            out.loc[idx] = _city_to_state[city]
            audit['sibling'].append((wh, city, _city_to_state[city])); continue
        # Layer 3: override
        if wh in WAREHOUSE_STATE_OVERRIDES:
            out.loc[idx] = WAREHOUSE_STATE_OVERRIDES[wh]
            audit['override'].append((wh, WAREHOUSE_STATE_OVERRIDES[wh])); continue
        # Layer 4: unknown
        out.loc[idx] = 'Unknown'
        audit['unknown'].append(wh)
    return out, audit

def _print_state_audit(audit, label):
    print(f'\n[{label}] State resolution audit:')
    if audit['sibling']:
        print(f'  Sibling-city resolved ({len(audit["sibling"])}):')
        # dedupe: show distinct (wh, state) pairs only once
        _seen = set()
        for wh, city, st in audit['sibling']:
            if (wh, st) not in _seen:
                print(f"    {wh:<40} (city='{city}') -> {st}")
                _seen.add((wh, st))
    if audit['override']:
        print(f'  Override-resolved ({len(audit["override"])}):')
        _seen = set()
        for wh, st in audit['override']:
            if (wh, st) not in _seen:
                print(f"    {wh:<40} -> {st} [override]")
                _seen.add((wh, st))
    if audit['unknown']:
        _unique_unknown = sorted(set(audit['unknown']))
        print(f'  *** UNRESOLVED ({len(_unique_unknown)} distinct warehouses) — set to state="Unknown" ***')
        for wh in _unique_unknown:
            print(f'    {wh}')
        print('  ACTION: Add to WAREHOUSE_STATE_OVERRIDES (Cell 3) or fix SERP_WAREHOUSES.[State Province].')

df_tx['state'], _state_audit = resolve_state_series(df_tx['tech_warehouse'], df_tx['state'], df_hier)
_print_state_audit(_state_audit, 'df_tx')

# Remove R0/R1/DI/RN/etc region codes for states
_bogus = df_tx['state'].astype(str).str.match(r'^R\d|^DI$|^RN$', na=False)
if _bogus.any():
    _bogus_whs = sorted(df_tx.loc[_bogus,'tech_warehouse'].dropna().unique())
    print(f'\n*** WARNING: {_bogus.sum():,} rows still have region-code states. Warehouses: {_bogus_whs}')

_tot=df_tx['order_num'].nunique(); _n_field=(~df_tx['_unattributed']).sum()
_n_blank=df_tx['_is_blank_tech'].sum(); _n_unmatch=df_tx['_is_unmatched'].sum()
_n_ambig=df_tx['_is_ambiguous'].sum(); _n_intops=df_tx['_is_internal_ops'].sum(); _n_unattr=df_tx['_unattributed'].sum()
print(f'Matching complete. {_tot:,} tickets — field: {_n_field:,} ({_n_field/_tot*100:.1f}%) | dark: {_n_unattr:,} ({_n_unattr/_tot*100:.1f}%)')
print('\nState distribution (top 10):')
print(df_tx['state'].value_counts(dropna=False).head(10).to_string())


[df_tx] State resolution audit:
  Sibling-city resolved (8743):
    R05 Birmingham (Hoover)                  (city='Birmingham (Hoover)') -> AL
    RNW San Antonio WH 2                     (city='San Antonio WH 2') -> TX
    RNW San Antonio                          (city='San Antonio') -> TX
    R08 Gadsden (Rainbow City)               (city='Gadsden (Rainbow City)') -> AL
    R10 Winchester                           (city='Winchester') -> VA
    R10 Lorton                               (city='Lorton') -> VA
    RNW McAllen                              (city='McAllen') -> TX
    RNW Corpus Christi                       (city='Corpus Christi') -> TX
    RNW Laredo                               (city='Laredo') -> TX
    RNW Irving                               (city='Irving') -> TX
    R08 Enterprise (Dothan)                  (city='Enterprise (Dothan)') -> AL
    R08 Birmingham (Hoover)                  (city='Birmingham (Hoover)') -> AL
    RNW Del Rio                              (ci

## Cell 10 — VISIT DEDUPLICATION (EXCHANGE CONSOLIDATION) *(LIFTED VERBATIM from tech_workload v1.35.0 — re-sync if the main notebook changes)*

In [23]:
_KEY=['record_id','techfirstname','techlastname','completed_date']
# 'exchange pair' = same patient + tech + date with BOTH a Pickup and a Delivery row. Only the Pickup is the redundant half; 
# the Delivery represents the visit. Service tickets (and anything else) in the same group are independent work and must be kept.
_tt_set=df_tx.groupby(_KEY)['ticket_type'].transform(lambda s:('Pickup' in s.values)and('Delivery' in s.values))
df_tx['_is_exchange_pair']=_tt_set.fillna(False).astype(bool)
# Drop ONLY the Pickup row(s) inside an exchange pair; keep every other row.
df_tx['_keep'] = ~(df_tx['_is_exchange_pair'] & (df_tx['ticket_type']=='Pickup'))
df_visits=df_tx[df_tx['_keep']].copy()
# v1.33.0 (M4): only the Delivery half of an exchange pair becomes 'Exchange'.
# Service (or other) tickets sharing the (patient, tech, date) group are
# independent work and keep their own type — they were previously mislabeled
# 'Exchange' (row counts were unaffected; the visit-type MIX was wrong).
df_visits['visit_type']=np.where(df_visits['_is_exchange_pair']&(df_visits['ticket_type']=='Delivery'),'Exchange',df_visits['ticket_type'])
n_raw,n_visits=len(df_tx),len(df_visits); n_ex=int(df_visits['_is_exchange_pair'].sum())
print(f'Raw: {n_raw:,}  -> Visits: {n_visits:,}  ({(1-n_visits/n_raw)*100:.1f}% reduction)  Exchange pairs: {n_ex:,}')
print(df_visits['visit_type'].value_counts().to_string())
_ex_kept=df_visits[df_visits['_is_exchange_pair']].groupby(_KEY)['order_num'].count()
print('PASS: Exchange dedup validated.' if (_ex_kept>1).sum()==0 else f'WARNING: {(_ex_kept>1).sum():,} groups >1 kept row.')

Raw: 449,326  -> Visits: 419,720  (6.6% reduction)  Exchange pairs: 31,854
visit_type
Delivery    225375
Pickup      111885
Service      42370
Exchange     29889
Split         5570
Other         4631


## Cell 11.1 — REDELIVERY DATE RESCUE & WINDOW FILTER *(LIFTED VERBATIM from tech_workload v1.35.0 — re-sync if the main notebook changes)*

In [24]:
# Re-Delivery Report dates are VARCHAR (locked lesson: filter in Python, not SQL).
# Rescue rows TRY_CONVERT missed via a second pandas parse of the raw string,
# count what stays unparseable (dropped LOUDLY), refilter to the window, and
# build event_key = (orig_order, redelivery date) — the dedup unit everywhere.
print(f'df_redel raw rows (post-SQL filter): {len(df_redel):,}')
_df_redel_orig=df_redel.copy()
_rd_from_convert=pd.to_datetime(_df_redel_orig['rd_date'],errors='coerce')
_rd_from_raw    =pd.to_datetime(_df_redel_orig['rd_datetime_raw'],errors='coerce')
_best_date=_rd_from_convert.where(_rd_from_convert.notna(),_rd_from_raw)
_df_redel_orig['_best_rd_date']=_best_date
_n_rescued=(_rd_from_convert.isna()&_rd_from_raw.notna()).sum()
if _n_rescued>0: print(f'  Rescued via raw parse: {_n_rescued:,}')
_n_unparseable = (_rd_from_convert.isna() & _rd_from_raw.isna()).sum()
if _n_unparseable>0:
    print(f'  *** WARNING: {_n_unparseable:,} redelivery rows have UNPARSEABLE Completion_DateTime')
    print(f'      These rows are DROPPED from all redelivery analytics.')
    print(f'      Sample unparseable raw values: {_df_redel_orig.loc[(_rd_from_convert.isna() & _rd_from_raw.isna()),"rd_datetime_raw"].dropna().astype(str).head(5).tolist()}')
_mask=(_df_redel_orig['_best_rd_date']>=pd.Timestamp(FILTER_START))&(_df_redel_orig['_best_rd_date']<=pd.Timestamp(FILTER_END))
df_redel=_df_redel_orig[_mask].copy()
print(f'  Window filter: {len(df_redel):,} kept / {(~_mask).sum():,} dropped (incl. {_n_unparseable:,} unparseable)')
df_redel['rd_date']=df_redel['_best_rd_date'].dt.date; df_redel.drop(columns=['_best_rd_date'],inplace=True)
df_redel['event_key'] = (df_redel['orig_order_num'].astype(str).str.strip()
                          + '|' + df_redel['rd_date'].astype(str))
print(f'After filter: {len(df_redel):,} rows  |  {df_redel["event_key"].nunique():,} unique (orig_order, rd_date) events')

def _parse_products(raw):
    if not raw or not isinstance(raw,str) or raw.strip()=='': return ['Unknown / Blank']
    items=[p.strip() for p in raw.replace('\r\n','\n').replace('\r','\n').split('\n') if p.strip()]
    return items or ['Unknown / Blank']

df_redel_exploded=(df_redel.copy().assign(product_list=lambda d:d['rd_products'].apply(_parse_products)).explode('product_list').rename(columns={'product_list':'product'}).reset_index(drop=True))
df_redel_exploded['product']=df_redel_exploded['product'].str.strip()
df_redel_exploded.loc[df_redel_exploded['product'].isna()|(df_redel_exploded['product']==''),'product']='Unknown / Blank'


df_redel raw rows (post-SQL filter): 187,737
  Window filter: 187,737 kept / 0 dropped (incl. 0 unparseable)
After filter: 187,737 rows  |  118,464 unique (orig_order, rd_date) events


## Cell 11.2 — LINK REDELIVERIES TO ORIGINATING TICKETS *(LIFTED VERBATIM from tech_workload v1.35.0 — re-sync if the main notebook changes)*

In [25]:
df_tx['order_num']=df_tx['order_num'].astype(str).str.strip()
df_redel_exploded['orig_order_num']=df_redel_exploded['orig_order_num'].astype(str).str.strip()
_order_wh_counts=df_tx.groupby(['order_num','tech_warehouse'],dropna=False).size().reset_index(name='_n')
_modal_wh=(_order_wh_counts.sort_values(['order_num','_n','tech_warehouse'],ascending=[True,False,True]).drop_duplicates(subset=['order_num'],keep='first')[['order_num','tech_warehouse']])
_tx_keys=(df_tx[['order_num','tech_warehouse','region','vp','state','metro','techfirstname','techlastname','delivery_year','delivery_month','schedule_period']].merge(_modal_wh,on=['order_num','tech_warehouse'],how='inner').drop_duplicates(subset=['order_num']).rename(columns={'tech_warehouse':'tx_warehouse','region':'tx_region','vp':'tx_vp','state':'tx_state','metro':'tx_metro','techfirstname':'tx_techfirstname','techlastname':'tx_techlastname','delivery_year':'tx_delivery_year','delivery_month':'tx_delivery_month','schedule_period':'tx_schedule_period'}))
redel_linked=df_redel_exploded.merge(_tx_keys,left_on='orig_order_num',right_on='order_num',how='inner')
print(f'Unlinked: {len(df_redel_exploded)-len(redel_linked):,}  Linked: {len(redel_linked):,}')
# v1.33.0 (M2): reconcile UNLINKED redeliveries by month (of the redelivery date).
# These are events whose originating order is outside the ticket window (e.g. a
# Jan-2025 redelivery of a Dec-2024 original) or under a non-included Reason.
# They are EXCLUDED from every linked redelivery metric — the early months of the
# window undercount by design. This table makes the exclusion auditable in Excel
# (sheet 'Redel_Unlinked_Monthly'); footnote any board chart that spans the edge.
_unlinked = df_redel_exploded[~df_redel_exploded['orig_order_num'].isin(set(_tx_keys['order_num']))].copy()
_unlinked['_rd_dt'] = pd.to_datetime(_unlinked['rd_date'], errors='coerce')
tbl_redel_unlinked_monthly = (_unlinked
    .assign(delivery_year=_unlinked['_rd_dt'].dt.year, delivery_month=_unlinked['_rd_dt'].dt.month)
    .groupby(['delivery_year','delivery_month'], dropna=False, as_index=False)
    .agg(unlinked_events=('event_key','nunique'), unlinked_items=('product','count')))
tbl_redel_unlinked_monthly['period'] = (tbl_redel_unlinked_monthly['delivery_year'].astype('Int64').astype(str)
    + '-' + tbl_redel_unlinked_monthly['delivery_month'].astype('Int64').astype(str).str.zfill(2))
print(f'  Unlinked reconciliation: {tbl_redel_unlinked_monthly["unlinked_events"].sum():,} events across '
      f'{len(tbl_redel_unlinked_monthly)} month(s) — exported to Redel_Unlinked_Monthly.')
for _src,_dst in [('tx_warehouse','tech_warehouse'),('tx_region','region'),('tx_vp','vp'),('tx_state','state'),('tx_metro','metro'),('tx_techfirstname','techfirstname'),('tx_techlastname','techlastname'),('tx_delivery_year','delivery_year'),('tx_delivery_month','delivery_month'),('tx_schedule_period','schedule_period')]:
    redel_linked[_dst]=redel_linked[_src]
redel_linked.drop(columns=[c for c in redel_linked.columns if c.startswith('tx_')],inplace=True,errors='ignore')

redel_linked['period']=(redel_linked['delivery_year'].astype(int).astype(str)+'-'+redel_linked['delivery_month'].astype(int).astype(str).str.zfill(2))
tbl_redel_product=(redel_linked.groupby('product').agg(
    redelivery_count=('event_key','nunique'),
    warehouses=('tech_warehouse',lambda x:x.nunique())
).reset_index().sort_values('redelivery_count',ascending=False))
tbl_redel_monthly=(redel_linked.groupby(['delivery_year','delivery_month']).agg(
    redelivery_count=('event_key','nunique'),
    redelivery_items=('product','count')   # count of product rows (items returned)
).reset_index().sort_values(['delivery_year','delivery_month']))
tbl_redel_monthly['period']=tbl_redel_monthly['delivery_year'].astype(str)+'-'+tbl_redel_monthly['delivery_month'].astype(str).str.zfill(2)
_tx_monthly_total = df_tx.groupby(['delivery_year','delivery_month'],dropna=False,as_index=False).agg(total_tickets=('order_num','nunique'))
tbl_redel_monthly = tbl_redel_monthly.merge(_tx_monthly_total,on=['delivery_year','delivery_month'],how='left')
tbl_redel_monthly['redel_pct_of_tickets'] = (tbl_redel_monthly['redelivery_count']/tbl_redel_monthly['total_tickets'].replace(0,np.nan)*100).round(2)
display(tbl_redel_product.head(10))
print(f'Redelivery exploded rows: {len(redel_linked):,}')

Unlinked: 62,482  Linked: 443,115
  Unlinked reconciliation: 10,476 events across 19 month(s) — exported to Redel_Unlinked_Monthly.


,product,redelivery_count,warehouses
240,"Oxygen Cylinder, E Tank(100)",17811,90
165,Full Electric Hospital Bed(294),10672,89
226,Over Bed Table(303),10082,89
237,"Oxygen Concentrator, 5 Liter(305)",9889,87
320,Pressure Prevention Foam Mattress(314),9181,88
37,7' Oxygen Tubing(3),8125,87
231,"Oxygen Cannula, Adult(16)",7837,87
472,Water Trap(40),7815,87
185,Humidifier Bottle Adapter(11),7460,86
246,"Oxygen Gauge, Portable(309)",7206,85


Redelivery exploded rows: 443,115


## Cell 12 — LOST EQUIPMENT PREP & WINDOW FILTER *(LIFTED VERBATIM from tech_workload v1.35.0 — re-sync if the main notebook changes)*

In [26]:
# ── Lost equipment ────────────────────────────────────────────────────────────
df_lost_raw['lost_date']=pd.to_datetime(df_lost_raw['lost_date_raw'],errors='coerce')
df_lost_raw['asset_tag']=pd.to_numeric(df_lost_raw['asset_tag'],errors='coerce').astype('Int64')
df_lost_raw['asset_tag_str']=df_lost_raw['asset_tag'].astype(str).replace('<NA>',pd.NA)
df_lost_raw['state'], _lost_state_audit = resolve_state_series(
    df_lost_raw['tech_warehouse'], df_lost_raw['state'], df_hier)
_print_state_audit(_lost_state_audit, 'df_lost_raw')
df_lost_raw['metro']=df_lost_raw['tech_warehouse'].apply(assign_metro)

_lost_filt=df_lost_raw[
    df_lost_raw['lost_date'].notna()&
    (df_lost_raw['lost_date']>=pd.Timestamp(FILTER_START))&
    (df_lost_raw['lost_date']<=pd.Timestamp(FILTER_END))
].copy()
print(f'\nLost equipment in window ({FILTER_START} to {FILTER_END}): {len(_lost_filt):,}')
if len(_lost_filt)==0:
    print('  *** WARNING: zero lost-equipment rows in window.')
    print(f'      df_lost_raw has {len(df_lost_raw):,} rows total but NONE fall between')
    print(f'      {FILTER_START} and {FILTER_END}. Lost-equipment sheets will be empty.')
    print(f'      Likely cause: ATI.Lost_Date feed has not been updated, or FILTER_START')
    print(f'      is set later than any lost event in the table.')
_lost_filt['lost_year'] =_lost_filt['lost_date'].dt.year.fillna(0).astype(int)
_lost_filt['lost_month']=_lost_filt['lost_date'].dt.month.fillna(0).astype(int)
_lost_filt['lost_cost_last_price']=pd.to_numeric(_lost_filt['lost_cost_last_price'],errors='coerce').fillna(0.0)

tbl_lost_by_wh=(_lost_filt.groupby(['tech_warehouse','region','vp','state','metro','lost_year','lost_month'],dropna=False).agg(lost_asset_count=('asset_tag','nunique'),lost_asset_cost=('lost_cost_last_price','sum'),lost_product_types=('product_name','nunique'),lost_products_list=('product_name',lambda x:' | '.join(sorted(x.dropna().unique())))).reset_index().sort_values('lost_asset_cost',ascending=False))
tbl_lost_by_wh=tbl_lost_by_wh.merge(df_adc[['warehouse','yr','mo','adc','pt_days']],left_on=['tech_warehouse','lost_year','lost_month'],right_on=['warehouse','yr','mo'],how='left').drop(columns=['warehouse','yr','mo'],errors='ignore')
tbl_lost_by_wh['lost_cost_per_adc']   =(tbl_lost_by_wh['lost_asset_cost']/tbl_lost_by_wh['adc'].replace(0,np.nan)).round(2)
tbl_lost_by_wh['lost_cost_per_pt_day']=(tbl_lost_by_wh['lost_asset_cost']/tbl_lost_by_wh['pt_days'].replace(0,np.nan)).round(4)

_lost_filt['_bid_str']=_lost_filt['bill_to_id'].astype(str).str.strip()
_lost_patient_ids=set(_lost_filt['_bid_str'].replace('',pd.NA).replace('nan',pd.NA).replace('None',pd.NA).replace('<NA>',pd.NA).replace('0',pd.NA).dropna().unique())
print(f'Unique patients in lost-asset set: {len(_lost_patient_ids):,}')



[df_lost_raw] State resolution audit:

Lost equipment in window (2025-01-01 to 2026-08-13): 0
  *** WARNING: zero lost-equipment rows in window.
      df_lost_raw has 0 rows total but NONE fall between
      2025-01-01 and 2026-08-13. Lost-equipment sheets will be empty.
      Likely cause: ATI.Lost_Date feed has not been updated, or FILTER_START
      is set later than any lost event in the table.
Unique patients in lost-asset set: 0


## Cell 13 — WEEKLY TECHNICIAN PRODUCTIVITY BY ENTITY (new)

In [27]:
# ─────────────────────────────────────────────────────────────────────────────
# WEEKLY TECHNICIAN PRODUCTIVITY BY ENTITY (adapted from v1.35.0 Cell 13c)
# Sun–Sat weeks (FLSA-aligned). Weekday tickets only. Ratio-of-pooled-sums
# (locked rule #2). Grains produced: warehouse, metro, VP, company.
# ─────────────────────────────────────────────────────────────────────────────
_win_start = pd.Timestamp(FILTER_START)
_win_end   = pd.Timestamp(AS_OF_DATE)

_wk_src = df_tx[(~df_tx['_unattributed']) & (df_tx['schedule_period'] == 'Weekday')].copy()
_wk_src['week_start'] = (_wk_src['completed_date']
    - pd.to_timedelta((_wk_src['completed_date'].dt.dayofweek + 1) % 7, unit='D')).dt.normalize()
_wk_src['_tech_key'] = _wk_src['techfirstname'].str.strip() + '|' + _wk_src['techlastname'].str.strip()

_week_index = pd.DataFrame({'week_start': sorted(_wk_src['week_start'].dropna().unique())})
def _weekdays_in_week(ws):
    lo, hi = max(ws, _win_start), min(ws + pd.Timedelta(days=6), _win_end)
    if lo > hi: return 0
    d = pd.date_range(lo, hi, freq='D')
    return int((d.dayofweek < 5).sum())
_week_index['weekday_days_in_week'] = _week_index['week_start'].apply(_weekdays_in_week)
_week_index['is_partial_week'] = _week_index['weekday_days_in_week'] < 5
_week_index['week_label'] = _week_index['week_start'].dt.strftime('%Y-%m-%d')

# Tech-week base at (tech, warehouse, metro, vp, week) — one row per tech per warehouse.
_techday_wk = (_wk_src.groupby(['techfirstname','techlastname','_tech_key','tech_warehouse',
                                'vp','state','metro','week_start'], dropna=False, as_index=False)
    .agg(total_tickets=('order_num','nunique'), active_weekdays=('completed_date','nunique')))

def _dash_weekly(gcols):
    """Pooled weekly rollup for any entity grain ([]=company)."""
    keys = gcols + ['week_start']
    agg = (_techday_wk.groupby(keys, dropna=False, as_index=False)
        .agg(total_tickets=('total_tickets','sum'),
             total_active_tech_days=('active_weekdays','sum'),
             tech_count=('_tech_key','nunique'))
        .merge(_week_index, on='week_start', how='left'))
    agg['tickets_per_active_day_per_tech'] = (agg['total_tickets']
        / agg['total_active_tech_days'].replace(0, np.nan)).round(3)
    _cal_denom = (agg['tech_count'] * agg['weekday_days_in_week']).replace(0, np.nan)
    agg['weekday_daily_tickets_per_tech'] = (agg['total_tickets'] / _cal_denom).round(3)
    agg = agg.sort_values((gcols if gcols else []) + ['week_start']).reset_index(drop=True)
    # rolling 4-week average of the headline metric (last 4 observed weeks, min 2)
    m = 'tickets_per_active_day_per_tech'
    if gcols:
        agg['rolling_4wk_avg'] = (agg.groupby(gcols, dropna=False)[m]
            .transform(lambda s: s.rolling(DASH_ROLL_WEEKS, min_periods=2).mean()).round(3))
    else:
        agg['rolling_4wk_avg'] = agg[m].rolling(DASH_ROLL_WEEKS, min_periods=2).mean().round(3)
    return agg

dash_wk_prod_wh    = _dash_weekly(['tech_warehouse','vp','state','metro'])
dash_wk_prod_metro = _dash_weekly(['metro'])
dash_wk_prod_vp    = _dash_weekly(['vp'])
dash_wk_prod_co    = _dash_weekly([])
print(f'Weekly productivity: wh={len(dash_wk_prod_wh):,} metro={len(dash_wk_prod_metro):,} '
      f'vp={len(dash_wk_prod_vp):,} co={len(dash_wk_prod_co):,} ({len(_week_index)} weeks)')


Weekly productivity: wh=5,347 metro=340 vp=485 co=85 (85 weeks)


## Cell 14 — MONTHLY LOST EQUIPMENT BY ENTITY (new)

In [28]:
# ─────────────────────────────────────────────────────────────────────────────
# MONTHLY LOST EQUIPMENT BY ENTITY (source: _lost_filt from the lifted Cell 17.2)
# Month = month the asset was MARKED lost (ATI.Lost_Date), matching the main
# notebook's convention. Monthly (not weekly): warehouse-week lost counts are
# mostly 0–2 and would chart as noise.
# ─────────────────────────────────────────────────────────────────────────────
def _dash_lost_monthly(gcols):
    keys = gcols + ['lost_year','lost_month']
    agg = (_lost_filt.groupby(keys, dropna=False, as_index=False)
        .agg(lost_asset_count=('asset_tag','nunique'),
             lost_asset_cost=('lost_cost_last_price','sum'),
             lost_product_types=('product_name','nunique')))
    agg['period'] = (agg['lost_year'].astype(int).astype(str) + '-'
                     + agg['lost_month'].astype(int).astype(str).str.zfill(2))
    agg['lost_asset_cost'] = agg['lost_asset_cost'].round(2)
    return agg.sort_values((gcols if gcols else []) + ['lost_year','lost_month']).reset_index(drop=True)

dash_mo_lost_wh    = _dash_lost_monthly(['tech_warehouse','vp','state','metro'])
dash_mo_lost_metro = _dash_lost_monthly(['metro'])
dash_mo_lost_vp    = _dash_lost_monthly(['vp'])
dash_mo_lost_co    = _dash_lost_monthly([])
print(f'Monthly lost: wh={len(dash_mo_lost_wh):,} metro={len(dash_mo_lost_metro):,} '
      f'vp={len(dash_mo_lost_vp):,} co={len(dash_mo_lost_co):,}')


Monthly lost: wh=0 metro=0 vp=0 co=0


## Cell 15 — MONTHLY REDELIVERIES BY ENTITY (new)

In [29]:
# ─────────────────────────────────────────────────────────────────────────────
# MONTHLY REDELIVERIES BY ENTITY (source: redel_linked from the lifted Cell 16.2)
# Month = ORIGINATING ticket month (delivery_year/month from the tx merge),
# matching the main notebook's 16b tables so the two reports reconcile.
# Rate denominator = attributed tickets for the SAME entity+month (grain rule).
# Entity attribution = the originating ticket's warehouse/VP/metro (tx_ fields).
# ─────────────────────────────────────────────────────────────────────────────
_rl = redel_linked.copy()
_rl['metro_e'] = _rl['tx_metro']; _rl['vp_e'] = _rl['tx_vp']
_rl['wh_e'] = _rl['tx_warehouse']; _rl['state_e'] = _rl['tx_state']
_rl['yr'] = _rl['tx_delivery_year'].astype('Int64'); _rl['mo'] = _rl['tx_delivery_month'].astype('Int64')

_txa = df_tx[~df_tx['_unattributed']].copy()
def _dash_redel_monthly(ent_map):
    """ent_map: {entity_col_in_rl: matching df_tx col} — e.g. {'wh_e':'tech_warehouse'}."""
    rl_keys = list(ent_map.keys()) + ['yr','mo']
    agg = (_rl.groupby(rl_keys, dropna=False, as_index=False)
        .agg(redelivery_count=('event_key','nunique'), redelivery_items=('product','count')))
    tx_keys = list(ent_map.values()) + ['delivery_year','delivery_month']
    tx_m = (_txa.groupby(tx_keys, dropna=False, as_index=False)
        .agg(total_tickets=('order_num','nunique')))
    tx_m.columns = rl_keys + ['total_tickets']
    agg = agg.merge(tx_m, on=rl_keys, how='left')
    agg['redel_per_100_tickets'] = (agg['redelivery_count']
        / agg['total_tickets'].replace(0, np.nan) * 100).round(2)
    agg['period'] = (agg['yr'].astype(str) + '-' + agg['mo'].astype(str).str.zfill(2))
    return (agg.rename(columns=dict(zip(ent_map.keys(), ent_map.values())))
               .sort_values(list(ent_map.values()) + ['yr','mo']).reset_index(drop=True))

dash_mo_redel_wh    = _dash_redel_monthly({'wh_e':'tech_warehouse','vp_e':'vp','state_e':'state','metro_e':'metro'})
dash_mo_redel_metro = _dash_redel_monthly({'metro_e':'metro'})
dash_mo_redel_vp    = _dash_redel_monthly({'vp_e':'vp'})
dash_mo_redel_co    = _dash_redel_monthly({})
print(f'Monthly redeliveries: wh={len(dash_mo_redel_wh):,} metro={len(dash_mo_redel_metro):,} '
      f'vp={len(dash_mo_redel_vp):,} co={len(dash_mo_redel_co):,}')


KeyError: 'tx_metro'

## Cell 16 — STOCK OUTS (CONFIG-GATED — SOURCE PENDING) (new)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STOCK OUTS — CONFIG-GATED (v1.0.0)
# The stock-out source table has not been provided yet. Until STOCKOUT_TABLE is
# set in the config cell, this cell emits empty frames and every downstream
# panel/sheet displays "not configured" — deliberately loud, never proxied.
# ─────────────────────────────────────────────────────────────────────────────
dash_mo_stockout_wh = pd.DataFrame(); dash_mo_stockout_metro = pd.DataFrame()
dash_mo_stockout_vp = pd.DataFrame(); dash_mo_stockout_co = pd.DataFrame()
if STOCKOUT_TABLE is None:
    print('*** STOCK OUTS NOT CONFIGURED: set STOCKOUT_TABLE + STOCKOUT_COLS in the config')
    print('    cell once the source table/report name is provided. Panels will show a notice.')
else:
    # Template query — adjust to the real schema when the table name arrives.
    _so = run_query(f"""
SELECT {STOCKOUT_COLS['warehouse']} AS tech_warehouse,
       {STOCKOUT_COLS['date']}      AS so_date_raw,
       {STOCKOUT_COLS['product']}   AS product
       {',' + STOCKOUT_COLS['qty'] + ' AS qty' if STOCKOUT_COLS.get('qty') else ''}
FROM {STOCKOUT_TABLE} WITH (NOLOCK)
""", 'Stock outs')
    _so['so_date'] = pd.to_datetime(_so['so_date_raw'], errors='coerce')
    _n_bad = int(_so['so_date'].isna().sum())
    if _n_bad: print(f'  *** {_n_bad:,} stock-out rows with unparseable dates DROPPED.')
    _so = _so[_so['so_date'].notna()
              & (_so['so_date'] >= pd.Timestamp(FILTER_START))
              & (_so['so_date'] <= pd.Timestamp(FILTER_END))].copy()
    _so['state'], _ = resolve_state_series(_so['tech_warehouse'], pd.Series(pd.NA, index=_so.index), df_hier)
    _so['metro'] = _so['tech_warehouse'].apply(assign_metro)
    _so = _so.merge(df_hier[['tech_warehouse','vp']].drop_duplicates('tech_warehouse'),
                    on='tech_warehouse', how='left')
    _so['so_year'] = _so['so_date'].dt.year; _so['so_month'] = _so['so_date'].dt.month
    def _so_monthly(gcols):
        agg = (_so.groupby(gcols + ['so_year','so_month'], dropna=False, as_index=False)
               .agg(stockout_events=('so_date','count'),
                    distinct_products=('product','nunique')))
        agg['period'] = (agg['so_year'].astype(int).astype(str) + '-'
                         + agg['so_month'].astype(int).astype(str).str.zfill(2))
        return agg.sort_values((gcols if gcols else []) + ['so_year','so_month']).reset_index(drop=True)
    dash_mo_stockout_wh    = _so_monthly(['tech_warehouse','vp','state','metro'])
    dash_mo_stockout_metro = _so_monthly(['metro'])
    dash_mo_stockout_vp    = _so_monthly(['vp'])
    dash_mo_stockout_co    = _so_monthly([])
    print(f'Stock outs: {len(_so):,} events in window.')


## Cell 17 — TOP & BOTTOM TECHNICIANS PER GROUPING (new)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TOP & BOTTOM TECHNICIANS PER GROUPING (v1.0.0)
# Ranking metric: pooled tickets-per-active-day (headline). Eligibility:
# >= DASH_MIN_ACTIVE_DAYS active weekdays inside the grouping. Redelivery stats
# joined at the SAME grain as the pool (grain-discipline rule). Top and bottom
# lists are kept disjoint: a tech can never appear on both sides.
# ─────────────────────────────────────────────────────────────────────────────
_redel_by_tech = (redel_linked[redel_linked['techfirstname'].fillna('').str.strip().ne('')]
    .groupby(['techfirstname','techlastname','tech_warehouse'], dropna=False, as_index=False)
    .agg(redelivery_count=('event_key','nunique')))

def _dash_topbottom(gcols, level_name):
    """Pooled per-tech stats within each entity of `gcols`; returns top/bottom N."""
    keys = ['techfirstname','techlastname','tech_warehouse'] + [c for c in gcols
            if c not in ('techfirstname','techlastname','tech_warehouse')]
    pool = (_techday_wk.groupby(keys, dropna=False, as_index=False)
        .agg(total_tickets=('total_tickets','sum'),
             active_weekdays=('active_weekdays','sum'),
             weeks_active=('week_start','nunique')))
    pool = pool.merge(_redel_by_tech, on=['techfirstname','techlastname','tech_warehouse'], how='left')
    pool['redelivery_count'] = pool['redelivery_count'].fillna(0).astype(int)
    pool['tickets_per_active_day'] = (pool['total_tickets']
        / pool['active_weekdays'].replace(0, np.nan)).round(3)
    pool['redel_per_100_tickets'] = (pool['redelivery_count']
        / pool['total_tickets'].replace(0, np.nan) * 100).round(2)
    pool = pool[pool['active_weekdays'] >= DASH_MIN_ACTIVE_DAYS].copy()
    out = []
    ents = pool[gcols].drop_duplicates().itertuples(index=False) if gcols else [None]
    for ent in ents:
        sub = pool
        if ent is not None:
            for c, v in zip(gcols, ent):
                sub = sub[(sub[c] == v) | (sub[c].isna() & pd.isna(v))]
        sub = sub.sort_values(['tickets_per_active_day','total_tickets'],
                              ascending=[False, False]).reset_index(drop=True)
        n = min(DASH_TOP_N, len(sub))
        top = sub.head(n).copy(); top['rank_group'] = 'Top'
        # disjoint: bottom drawn only from rows not already in top
        bot = sub.iloc[n:].tail(min(DASH_TOP_N, max(0, len(sub) - n))).copy()
        bot['rank_group'] = 'Bottom'
        blk = pd.concat([top, bot], ignore_index=True)
        blk['grouping_level'] = level_name
        out.append(blk)
    if not out: return pd.DataFrame()
    res = pd.concat(out, ignore_index=True)
    _stat_cols = ['grouping_level','rank_group','techfirstname','techlastname','tech_warehouse',
                  'total_tickets','active_weekdays','weeks_active','tickets_per_active_day',
                  'redelivery_count','redel_per_100_tickets']
    return res[[c for c in gcols if c not in _stat_cols] + _stat_cols]

dash_tb_wh    = _dash_topbottom(['tech_warehouse'], 'Warehouse')  # tech's own wh = the entity
dash_tb_metro = _dash_topbottom(['metro'], 'Metro')
dash_tb_vp    = _dash_topbottom(['vp'], 'VP')
dash_tb_co    = _dash_topbottom([], 'Company')
print(f'Top/Bottom lists: wh={len(dash_tb_wh):,} metro={len(dash_tb_metro):,} '
      f'vp={len(dash_tb_vp):,} co={len(dash_tb_co):,} rows '
      f'(N={DASH_TOP_N}/side, floor={DASH_MIN_ACTIVE_DAYS} active days)')


## Cell 18 — DASHBOARD PAGES → PDF (+ PNGs for company/VP) (new)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# DASHBOARD PAGE RENDERER (v1.0.0)
# One landscape page per entity — company, each VP, each metro, each warehouse —
# compiled into a single PDF. Layout: (1) weekly productivity + rolling 4-week
# average, (2) monthly lost equipment (bars = assets, line = cost), (3) monthly
# redeliveries (bars = events, line = per-100-tickets; stock-outs overlay when
# configured), (4) top/bottom technician table. PNGs additionally saved for the
# company and each VP (the pages leadership circulates most).
# ─────────────────────────────────────────────────────────────────────────────
from matplotlib.backends.backend_pdf import PdfPages

def _panel_weekly(ax, wk, title):
    if wk is None or wk.empty:
        ax.text(0.5, 0.5, 'No weekly data', ha='center', va='center'); ax.set_title(title); return
    wk = wk.sort_values('week_start')
    _pf = wk['is_partial_week'].fillna(False).values
    ax.plot(wk['week_start'], wk['tickets_per_active_day_per_tech'],
            color=PALETTE['attributed'], lw=1.3, marker='o', ms=3.5, alpha=0.75, label='Weekly actual')
    # partial weeks re-marked hollow so a truncated week isn't read as a dip
    _p = wk[_pf]
    if len(_p): ax.plot(_p['week_start'], _p['tickets_per_active_day_per_tech'], ls='none',
                        marker='o', ms=5, mfc='white', mec=PALETTE['attributed'])
    ax.plot(wk['week_start'], wk['rolling_4wk_avg'], color='black', lw=2.2, ls='--',
            label=f'{DASH_ROLL_WEEKS}wk avg')
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.tick_params(labelsize=7); ax.grid(axis='y', ls='--', alpha=0.3); ax.set_axisbelow(True)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
    ax.legend(fontsize=6, loc='best')

def _panel_monthly_dual(ax, df, bar_col, line_col, title, bar_lbl, line_lbl, line_fmt='{:.0f}'):
    if df is None or df.empty:
        ax.text(0.5, 0.5, 'No data in window', ha='center', va='center'); ax.set_title(title); return
    df = df.sort_values('period')
    x = np.arange(len(df))
    ax.bar(x, df[bar_col].values, color=PALETTE['attributed'], alpha=0.75, label=bar_lbl)
    ax.set_xticks(x); ax.set_xticklabels(df['period'].values, rotation=60, fontsize=6)
    ax.tick_params(labelsize=7); ax.grid(axis='y', ls='--', alpha=0.3); ax.set_axisbelow(True)
    ax2 = ax.twinx()
    ax2.plot(x, df[line_col].values, color='#d62728', marker='D', ms=3.5, lw=1.6, label=line_lbl)
    ax2.tick_params(labelsize=7, colors='#d62728')
    ax.set_title(title, fontsize=9, fontweight='bold')
    h1,l1 = ax.get_legend_handles_labels(); h2,l2 = ax2.get_legend_handles_labels()
    ax.legend(h1+h2, l1+l2, fontsize=6, loc='best')

def _panel_topbottom(ax, tb, title):
    ax.axis('off'); ax.set_title(title, fontsize=9, fontweight='bold')
    if tb is None or tb.empty:
        ax.text(0.5, 0.5, f'No technicians clear the {DASH_MIN_ACTIVE_DAYS}-active-day floor',
                ha='center', va='center', fontsize=8); return
    cols = ['rank_group','techfirstname','techlastname','tech_warehouse',
            'total_tickets','active_weekdays','tickets_per_active_day','redel_per_100_tickets']
    d = tb[cols].copy()
    d.columns = ['','First','Last','Warehouse','Tickets','ActDays','Tkts/Day','Redel/100']
    t = ax.table(cellText=d.values, colLabels=d.columns, loc='center', cellLoc='center')
    t.auto_set_font_size(False); t.set_fontsize(6); t.scale(1, 1.15)
    for (r, c), cell in t.get_celld().items():
        if r == 0: cell.set_text_props(fontweight='bold'); cell.set_facecolor('#dbe5f1')
        elif d.iloc[r-1, 0] == 'Top': cell.set_facecolor('#eaf4ea')
        else: cell.set_facecolor('#fdecec')

def _entity_page(pdf, name, level, wk, lost, redel, so, tb, save_png=False):
    fig, axs = plt.subplots(2, 2, figsize=(15.5, 9.5))
    fig.suptitle(f'{level}: {name}   |   {FILTER_START} → {FILTER_END}   |   generated {RUN_DATE}',
                 fontsize=13, fontweight='bold')
    _panel_weekly(axs[0,0], wk, 'Technician Productivity — weekly tickets per active day per tech')
    _panel_monthly_dual(axs[0,1], lost, 'lost_asset_count', 'lost_asset_cost',
                        'Lost Equipment — monthly (bars = assets, line = $ cost)',
                        'Assets lost', 'Cost $')
    _panel_monthly_dual(axs[1,0], redel, 'redelivery_count', 'redel_per_100_tickets',
                        'Redeliveries — monthly (bars = events, line = per 100 tickets)',
                        'Redeliveries', 'Per 100 tickets')
    if so is not None and not so.empty:
        _so = so.sort_values('period')
        axs[1,0].plot(np.arange(len(_so)), _so['stockout_events'].values,
                      color='#7f7f7f', marker='s', ms=3, lw=1.2, label='Stock outs')
        axs[1,0].legend(fontsize=6, loc='best')
    else:
        axs[1,0].annotate('Stock outs: source not configured', xy=(0.99, 0.97),
                          xycoords='axes fraction', ha='right', va='top', fontsize=6.5,
                          style='italic', color='#777')
    _panel_topbottom(axs[1,1], tb, f'Top {DASH_TOP_N} / Bottom {DASH_TOP_N} Technicians '
                                    f'(≥{DASH_MIN_ACTIVE_DAYS} active days; tickets/active day)')
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    pdf.savefig(fig)
    if save_png: save_fig(fig, f'Dash_{level}_{re.sub(r"[^A-Za-z0-9]+","_",str(name))}')
    plt.show(); plt.close(fig)

def _sub(df, col, val): return df[df[col] == val] if (df is not None and not df.empty and col in df.columns) else pd.DataFrame()

_pdf_path = os.path.join(OUT_DIR, DASH_PDF_NAME)
with PdfPages(_pdf_path) as _pdf:
    # Company page first, then VPs, metros, warehouses (board reading order)
    _entity_page(_pdf, 'DME Express — All Operations', 'Company',
                 dash_wk_prod_co, dash_mo_lost_co, dash_mo_redel_co, dash_mo_stockout_co,
                 dash_tb_co, save_png=True)
    for _v in sorted(dash_wk_prod_vp['vp'].dropna().unique()):
        _entity_page(_pdf, _v, 'VP', _sub(dash_wk_prod_vp,'vp',_v), _sub(dash_mo_lost_vp,'vp',_v),
                     _sub(dash_mo_redel_vp,'vp',_v), _sub(dash_mo_stockout_vp,'vp',_v),
                     _sub(dash_tb_vp,'vp',_v), save_png=True)
    for _m in sorted(dash_wk_prod_metro['metro'].dropna().unique()):
        _entity_page(_pdf, _m, 'Metro', _sub(dash_wk_prod_metro,'metro',_m), _sub(dash_mo_lost_metro,'metro',_m),
                     _sub(dash_mo_redel_metro,'metro',_m), _sub(dash_mo_stockout_metro,'metro',_m),
                     _sub(dash_tb_metro,'metro',_m))
    for _w in sorted(dash_wk_prod_wh['tech_warehouse'].dropna().unique()):
        _entity_page(_pdf, _w, 'Warehouse', _sub(dash_wk_prod_wh,'tech_warehouse',_w),
                     _sub(dash_mo_lost_wh,'tech_warehouse',_w), _sub(dash_mo_redel_wh,'tech_warehouse',_w),
                     _sub(dash_mo_stockout_wh,'tech_warehouse',_w), _sub(dash_tb_wh,'tech_warehouse',_w))
print(f'Dashboard PDF written: {_pdf_path}')


## Cell 19 — EXCEL EXPORT & CONNECTION CLOSE (new)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# EXCEL EXPORT (v1.0.0) — one sheet per grouping × metric family, plus README.
# Lean writer (adapted from the main notebook's _ws): header styling, freeze
# panes, number formats, auto column width. No patient identifiers exported.
# ─────────────────────────────────────────────────────────────────────────────
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

_D_PCT = {'tickets_per_active_day_per_tech','weekday_daily_tickets_per_tech','rolling_4wk_avg',
          'tickets_per_active_day','redel_per_100_tickets','lost_asset_cost'}
_HDR_FILL = PatternFill('solid', fgColor='1F4E79'); _HDR_FONT = Font(color='FFFFFF', bold=True)

def _dws(wb, name, df, fcol=1):
    ws = wb.create_sheet(name[:31])
    if df is None or df.empty:
        ws.cell(1, 1, 'No data in window'); return
    d = df.copy()
    if 'week_start' in d.columns: d['week_start'] = pd.to_datetime(d['week_start']).dt.strftime('%Y-%m-%d')
    for j, col in enumerate(d.columns, 1):
        c = ws.cell(1, j, col); c.fill = _HDR_FILL; c.font = _HDR_FONT
        c.alignment = Alignment(horizontal='center')
    for i, row in enumerate(d.itertuples(index=False), 2):
        for j, (col, val) in enumerate(zip(d.columns, row), 1):
            if pd.isna(val): val = None
            elif isinstance(val, (np.integer,)): val = int(val)
            elif isinstance(val, (np.floating,)): val = float(val)
            cell = ws.cell(i, j, val)
            if col in _D_PCT: cell.number_format = '0.00'
            elif isinstance(val, int) and not isinstance(val, bool): cell.number_format = '#,##0'
    ws.freeze_panes = ws.cell(2, fcol + 1)
    for j, col in enumerate(d.columns, 1):
        _w = max(len(str(col)), int(d[col].astype(str).str.len().quantile(0.9)) if len(d) else 8)
        ws.column_dimensions[get_column_letter(j)].width = min(38, max(10, _w + 2))

wb = Workbook(); wb.remove(wb.active)
_readme = pd.DataFrame({'OpsDashboard README': [
    f'Generated {RUN_DATE} | window {FILTER_START} to {FILTER_END} | source notebook v1.0.0',
    'Productivity: weekly Sun–Sat, weekday tickets, tickets-per-active-day (pooled), rolling 4wk avg.',
    'Lost equipment: monthly by ATI Lost_Date month.',
    'Redeliveries: monthly by ORIGINATING ticket month (reconciles to main notebook 16b tables).',
    f'Stock outs: {"configured" if STOCKOUT_TABLE else "NOT CONFIGURED — source table pending"}.',
    f'Top/Bottom: {DASH_TOP_N} per side, ≥{DASH_MIN_ACTIVE_DAYS} active weekdays, ranked on tickets/active day; lists disjoint.',
    'Matching/attribution logic lifted verbatim from tech_workload v1.35.0 — keep in sync.']})
_dws(wb, 'README', _readme)
for _nm, _df, _fc in [
    ('Co_Prod_Weekly', dash_wk_prod_co, 1), ('VP_Prod_Weekly', dash_wk_prod_vp, 1),
    ('Metro_Prod_Weekly', dash_wk_prod_metro, 1), ('WH_Prod_Weekly', dash_wk_prod_wh, 1),
    ('Co_Lost_Monthly', dash_mo_lost_co, 1), ('VP_Lost_Monthly', dash_mo_lost_vp, 1),
    ('Metro_Lost_Monthly', dash_mo_lost_metro, 1), ('WH_Lost_Monthly', dash_mo_lost_wh, 1),
    ('Co_Redel_Monthly', dash_mo_redel_co, 1), ('VP_Redel_Monthly', dash_mo_redel_vp, 1),
    ('Metro_Redel_Monthly', dash_mo_redel_metro, 1), ('WH_Redel_Monthly', dash_mo_redel_wh, 1),
    ('TopBottom_Co', dash_tb_co, 2), ('TopBottom_VP', dash_tb_vp, 2),
    ('TopBottom_Metro', dash_tb_metro, 2), ('TopBottom_WH', dash_tb_wh, 2)]:
    _dws(wb, _nm, _df, _fc)
if STOCKOUT_TABLE is not None:
    for _nm, _df in [('Co_Stockout_Monthly', dash_mo_stockout_co), ('VP_Stockout_Monthly', dash_mo_stockout_vp),
                     ('Metro_Stockout_Monthly', dash_mo_stockout_metro), ('WH_Stockout_Monthly', dash_mo_stockout_wh)]:
        _dws(wb, _nm, _df)
_xlsx_path = os.path.join(OUT_DIR, DASH_XLSX_NAME)
wb.save(_xlsx_path)
print(f'Excel written: {_xlsx_path} ({len(wb.sheetnames)} sheets)')
try:
    sql_conn.close(); print('DB connection closed.')
except Exception:
    pass
